In [ ]:
# @title Ячейка 1: Быстрая подготовка окружения (~5-10 сек)
# @markdown Единый ноутбук этапов 1-12, версия 3.2.0. Код проекта лежит в пакете `support_router`
# @markdown Первая ячейка находит локальную копию или архив, иначе клонирует выбранную ветку GitHub вместе с датасетом.
import os, sys, subprocess, importlib.metadata, zipfile
from pathlib import Path
# Windows: дочерние процессы (тесты, приложение, проверки) пишут и читают в UTF-8, а не в кодировке системы.
os.environ.setdefault('PYTHONUTF8', '1')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
PROJECT_REPO = 'https://github.com/SofiCholoian888/support-ticket-router.git'
PROJECT_BRANCH = 'codex/router-review-fixes'  # Пустая строка выбирает ветку репозитория по умолчанию.
PROJECT_ARCHIVE = 'support-ticket-router.zip'

def find_project():
    here = Path.cwd().resolve()
    for folder in [here, here.parent, here/'support-ticket-router', Path('/content/support-ticket-router')]:
        if (folder/'src/support_router/__init__.py').exists():
            return folder.resolve()
    found = sorted(Path('support-ticket-router').glob('**/src/support_router/__init__.py')) if Path('support-ticket-router').exists() else []
    return found[0].parents[2].resolve() if found else None

PROJECT_DIR = find_project()
if PROJECT_DIR is None and PROJECT_REPO and not Path(PROJECT_ARCHIVE).is_file():
    branch_args = ['--branch', PROJECT_BRANCH] if PROJECT_BRANCH else []
    subprocess.check_call(['git', 'clone', '--depth', '1', *branch_args, PROJECT_REPO, 'support-ticket-router'])
    PROJECT_DIR = find_project()
if PROJECT_DIR is None:
    archive = Path(PROJECT_ARCHIVE)
    if not archive.exists() and 'google.colab' in sys.modules and os.environ.get('SUPPORT_NONINTERACTIVE') != '1':
        from google.colab import files
        print('Загрузите архив проекта support-ticket-router.zip')
        uploaded = [Path(name) for name in files.upload() if name.endswith('.zip')]
        archive = uploaded[0] if uploaded else archive
    if archive.exists():
        with zipfile.ZipFile(archive) as zf:
            zf.extractall('support-ticket-router')
        PROJECT_DIR = find_project()
if PROJECT_DIR is None:
    raise FileNotFoundError('Нет кода проекта: задайте PROJECT_REPO или положите support-ticket-router.zip рядом с ноутбуком.')
os.chdir(PROJECT_DIR)

required = {'scikit-learn': '1.8.0', 'gradio': '6.0.0', 'wordcloud': '1.9.6',
            'onnxruntime': '1.23.2', 'tokenizers': '0.22.1', 'prometheus-client': '0.26.0'}
missing = []
for name, version in required.items():
    try:
        if importlib.metadata.version(name) != version:
            missing.append(f'{name}=={version}')
    except importlib.metadata.PackageNotFoundError:
        missing.append(f'{name}=={version}')
for name in ['numpy', 'pandas', 'scipy', 'joblib', 'matplotlib', 'seaborn', 'pytest', 'fastapi', 'uvicorn', 'httpx', 'PyYAML']:
    try: importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: missing.append(name)
if missing:
    if 'sklearn' in sys.modules and any(x.startswith('scikit-learn') for x in missing):
        raise RuntimeError('Перезапустите среду и выполните ноутбук с первой ячейки: sklearn уже импортирован.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
RAW_DIR = PROJECT_DIR / 'data/raw'
PROCESSED_DIR = PROJECT_DIR / 'data/processed'
MODELS_DIR = PROJECT_DIR / 'models'
REPORTS_DIR = PROJECT_DIR / 'reports'
for p in [RAW_DIR, PROCESSED_DIR, MODELS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

for folder in [PROJECT_DIR, PROJECT_DIR / 'src']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
os.environ['PYTHONPATH'] = os.pathsep.join([str(PROJECT_DIR / 'src'), str(PROJECT_DIR)])
import importlib
import support_router as sc
from support_router import scenario_data, semantic_encoder, category_examples, manual_cases
# Семантический энкодер понадобится только при обучении semantic_* конфигураций.
# На Colab x86_64 он загружается при первом использовании; для офлайн smoke тестов
# допускается SUPPORT_SKIP_SEMANTIC_TESTS=1, но полный эксперимент требует загрузки.
import json, random, time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import f1_score, accuracy_score, classification_report
from threadpoolctl import threadpool_limits
SEED = sc.SEED
random.seed(SEED); np.random.seed(SEED)
pd.set_option('display.max_colwidth', 110)
sns.set_theme(style='whitegrid')
print('Версии:', sc.runtime_versions())
print('Проект:', PROJECT_DIR)
print('Признаки: только текст. category — цель; intent, flags, row_id, group — диагностика/разбиение.')
print('MiniLM (если используются semantic_*): локальный ONNX, первая загрузка ~24 МБ. Операционная автоотправка заблокирована до внешней проверки.')


**Этап 2. Загрузка и первичное изучение датасета**

Работаем с CSV **Bitext Sample Pre-built Customer Support Dataset for English (2020)**: 27 интентов и 11 категорий. Официальные значения **12 лингвистических flags** подтверждены карточкой [Bitext на Kaggle](https://www.kaggle.com/datasets/bitext/training-dataset-for-chatbotsvirtual-assistants/metadata). SLA и правила бизнес-приоритетов датасетом **не** определяются. Флаги — метаданные для диагностики, не признаки модели и не метки P1.

In [ ]:
# @title Ячейка 2: Загрузка датасета
from support_router.train import DEFAULT_RAW
raw_path = PROJECT_DIR / DEFAULT_RAW
if not raw_path.exists():
    candidates = []
    for folder in [PROJECT_DIR, Path.cwd(), Path('/content')]:
        if folder.exists():
            candidates += list(folder.glob('20000-Utterances-Training-dataset-for-chatbots-virtual-assistant-Bitext-sample*.csv'))
    candidates = sorted(set(p.resolve() for p in candidates))
    if len(candidates) == 1:
        raw_path = candidates[0]
    elif len(candidates) > 1:
        raise ValueError('Несколько исходных CSV: оставьте нужный файл или явно задайте raw_path.')
    elif 'google.colab' in sys.modules and os.environ.get('SUPPORT_NONINTERACTIVE') != '1':
        from google.colab import files
        uploaded = files.upload()
        matches = [Path(n) for n in uploaded if n.endswith('.csv')]
        if len(matches) != 1: raise ValueError('Загрузите один исходный CSV.')
        raw_path = matches[0]
    else:
        raise FileNotFoundError(f'Положите исходный CSV в {RAW_DIR} или текущую папку.')
raw = pd.read_csv(raw_path)
df = raw.copy()
required_columns = {'utterance', 'category', 'intent', 'flags'}
assert required_columns.issubset(df.columns), df.columns.tolist()
print('Размер:', df.shape, '| SHA256:', sc.sha256_file(raw_path))
display(df.head())
display(df.isna().sum().rename('missing'))


In [ ]:
# @title Ячейка 3: Примеры интентов по категориям
# @markdown Не все категории содержат несколько интентов: исключения LOIO показываются отдельно.

intents_by_cat = df.groupby('category')['intent'].unique().reset_index()
intents_by_cat['n_intents'] = intents_by_cat['intent'].apply(len)
display(intents_by_cat)
print(f"\nВсего категорий: {df['category'].nunique()}")
print(f"Всего интентов: {df['intent'].nunique()}")


In [ ]:
# @title Ячейка 4: Примеры флагов
# @markdown Официальный словарь Bitext 2020. Один запрос может иметь несколько флагов; пропуски не превращаются в буквы 'NAN'.

flags_series = df['flags']
parsed_flags = flags_series.map(sc.parse_linguistic_flags)
unique_flags = flags_series.dropna().astype(str).unique()
flag_dictionary = pd.DataFrame([
    {'flag': code, 'definition_from_Bitext': desc}
    for code, desc in sc.LINGUISTIC_FLAGS.items()
])
print(f'Уникальных непустых значений flags: {len(unique_flags)}')
print('Примеры (первые 30):', list(unique_flags[:30]))
display(flag_dictionary)
unknown_flag_codes = sorted({f for flags in parsed_flags for f in flags} - set(sc.LINGUISTIC_FLAGS))
missing_flag_count = int(parsed_flags.map(len).eq(0).sum())
print('Строк без распознанных буквенных флагов:', missing_flag_count)
print('Неизвестные документации коды:', unknown_flag_codes)
if unknown_flag_codes:
    print('ВНИМАНИЕ: неизвестные коды не интерпретируем без подтверждения первоисточника.')


In [ ]:
# @title Ячейка 5: Проверка плейсхолдеров
# Считаем уникальные строки с любым плейсхолдером, а не сумму пересекающихся regex.
placeholder_mask = df.utterance.fillna('').str.contains(sc._PLACEHOLDER, regex=True)
print(f'Строк с плейсхолдерами: {placeholder_mask.sum()} ({placeholder_mask.mean():.2%})')
display(df.loc[placeholder_mask, ['utterance']].head(10))


In [ ]:
# @title Ячейка 6: Дубликаты и пустые тексты
# @markdown Описательная статистика исходного корпуса, не метрика качества модели.

# Точные дубликаты по utterance
n_dup_text = df['utterance'].duplicated().sum()
# Полные дубликаты строк
n_dup_full = df.duplicated().sum()
# Пустые тексты
n_empty = (df['utterance'].astype(str).str.strip() == '').sum()

# Конфликты: один utterance — несколько category
conflicts = (df.groupby('utterance')['category'].nunique() > 1).sum()

print(f"Дубликаты по utterance: {n_dup_text}")
print(f"Полные дубликаты:       {n_dup_full}")
print(f"Пустые тексты:          {n_empty}")
print(f"Utterance с конфликтом категорий: {conflicts}")


In [ ]:
# @title Ячейка 7: Длины текстов
# @markdown Описательная статистика исходного корпуса, не метрика качества модели.

df['n_words'] = df['utterance'].astype(str).str.split().str.len()
df['n_chars'] = df['utterance'].astype(str).str.len()

display(df[['n_words','n_chars']].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['n_words'], bins=30, color='steelblue', edgecolor='k')
axes[0].set_title('Распределение числа слов в utterance')
axes[0].set_xlabel('Слов'); axes[0].set_ylabel('Частота')

axes[1].hist(df['n_chars'], bins=30, color='coral', edgecolor='k')
axes[1].set_title('Распределение числа символов')
axes[1].set_xlabel('Символов'); axes[1].set_ylabel('Частота')

plt.tight_layout(); plt.show()


In [ ]:
# @title Ячейка 8: Частоты отдельных флагов
# @markdown Одна строка учитывается не более одного раза для каждого флага. Флаги могут пересекаться.

from collections import Counter
all_chars = Counter(flag for codes in parsed_flags for flag in codes)
flag_freq = pd.DataFrame([
    {'flag_char':fc, 'meaning':sc.LINGUISTIC_FLAGS.get(fc, 'UNKNOWN'),
     'n_utterances':int(all_chars.get(fc, 0)),
     'share_of_rows':all_chars.get(fc, 0)/len(df)}
    for fc in sc.LINGUISTIC_FLAGS
]).sort_values('n_utterances', ascending=False).reset_index(drop=True)
display(flag_freq)
print('Сумма долей может быть >100%: у обращения может быть несколько языковых флагов.')


**Этап 3. EDA (Разведочный анализ)**

In [ ]:
# @title Ячейка 9: Распределение категорий и интентов
# @markdown Дисбаланс измеряем по фактическим данным. Он учитывается через macro-F1, recall и поддержку классов.

fig, axes = plt.subplots(1, 2, figsize=(14, 9))
cat_counts = df['category'].value_counts()
sns.barplot(y=cat_counts.index, x=cat_counts.values, ax=axes[0], color='steelblue')
axes[0].set_title('Распределение категорий (дисбаланс)')
axes[0].set_xlabel('Количество')
for i, v in enumerate(cat_counts.values):
    axes[0].text(v, i, f' {v}', va='center')

intent_counts = df['intent'].value_counts()
sns.barplot(y=intent_counts.index, x=intent_counts.values, ax=axes[1], color='coral')
axes[1].set_title('Распределение интентов')
axes[1].set_xlabel('Количество')

plt.tight_layout(); plt.show()

ratio = cat_counts.iloc[0] / cat_counts.iloc[-1]
print(f"Соотношение самого большого и самого маленького класса: {ratio:.2f}:1")


In [ ]:
# @title Ячейка 10: Дубликаты — подробно
# @markdown Промежуточная нормализация lower+strip; окончательная очистка будет в ячейке 20.

df_norm = df.copy()
df_norm['utterance_norm'] = df_norm['utterance'].astype(str).str.lower().str.strip()

# Конфликты на нормализованных данных
conflict_cat = (df_norm.groupby('utterance_norm')['category'].nunique() > 1).sum()
conflict_int = (df_norm.groupby('utterance_norm')['intent'].nunique() > 1).sum()
print(f"Нормализованные дубликаты: {df_norm['utterance_norm'].duplicated().sum()}")
print(f"Конфликты по категории: {conflict_cat}")
print(f"Конфликты по интенту:   {conflict_int}")

# Топ частых дубликатов
top_dup = df_norm['utterance_norm'].value_counts().head(10)
print("\nТоп-10 повторяющихся текстов:")
for text, cnt in top_dup.items():
    print(f"  {cnt}× {text[:80]}")


In [ ]:
# @title Ячейка 11: Топ слов и биграмм по категориям
# @markdown EDA CountVectorizer со стоп-словами; обучаемый TF-IDF дальше имеет собственный словарь.

from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(texts, n=1, top_k=10):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(texts)
    sums = vec.transform(texts).sum(axis=0).A1
    vocab = vec.get_feature_names_out()
    return sorted(zip(vocab, sums), key=lambda x: -x[1])[:top_k]

for cat in df['category'].unique()[:4]:  # первые 4 категории для краткости
    sub = df[df['category']==cat]['utterance']
    print(f"\n--- {cat} (n={len(sub)}) ---")
    print("Unigrams:", top_ngrams(sub, 1, 8))
    print("Bigrams: ", top_ngrams(sub, 2, 5))


In [ ]:
# @title Ячейка 12: 🌥 ОБЛАКО СЛОВ (добавлено)
# @markdown Описательная статистика исходного корпуса, не метрика качества модели.

from wordcloud import WordCloud

# Собираем весь текст в одну строку
full_text = " ".join(df['utterance'].astype(str).tolist())

# Строим облако
wc = WordCloud(
    width=1200, height=600,
    background_color='white',
    colormap='viridis',
    max_words=200,
    stopwords=None,          # стандартные стоп-слова WordCloud; это только иллюстрация
    collocations=False,
    random_state=SEED,
).generate(full_text)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Облако слов: самые частотные термы корпуса Bitext', fontsize=16)
plt.tight_layout(); plt.show()


In [ ]:
# @title Ячейка 13: 🌥 Облако слов по классам (добавлено)
# @markdown Описательная статистика исходного корпуса, не метрика качества модели.

from wordcloud import WordCloud

def plot_wordcloud(texts, title, ax):
    full = " ".join(texts.astype(str).tolist())
    if len(full.strip()) == 0:
        ax.set_title(title + " (пусто)"); ax.axis('off'); return
    wc = WordCloud(width=500, height=250, background_color='white',
                   colormap='plasma', max_words=60, collocations=False,
                   random_state=SEED).generate(full)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=11)

cats = df['category'].unique()
n = len(cats)
n_cols = 3
n_rows = (n + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()
for i, cat in enumerate(cats):
    sub = df[df['category']==cat]['utterance']
    plot_wordcloud(sub, f"{cat} (n={len(sub)})", axes[i])
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Облака слов по категориям', fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# @title Ячейка 14: Доля текстов с ключевыми маркерами
# @markdown str.contains(regex=False) считает подстроки; это EDA, а не правило приоритета или модель.

markers = {
    'payment': 'payment', 'error': 'error', 'account': 'account',
    'invoice': 'invoice', 'order': 'order', 'refund': 'refund',
    'reimbursement': 'reimbursement', 'delivery': 'delivery',
    'address': 'address', 'complaint': 'complaint',
}
texts = df['utterance'].astype(str).str.lower()
pen = pd.DataFrame({m: texts.str.contains(k, regex=False) for m, k in markers.items()})
pen['category'] = df['category'].values
penetration = pen.groupby('category').mean().round(3)
display(penetration)


In [ ]:
# @title Ячейка 15: Похожие классы (косинусное сходство)
# @markdown Сходство центроидов — лексическая диагностика, не измеренная вероятность ошибки.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfv = TfidfVectorizer(max_features=3000, stop_words='english')
X = tfv.fit_transform(df['utterance'].astype(str))

centroids = {}
for cat in df['category'].unique():
    idx = df['category'].values == cat
    centroids[cat] = np.asarray(X[idx].mean(axis=0)).ravel()

cats = list(centroids)
M = cosine_similarity(np.array([centroids[c] for c in cats]))

pairs = []
for i in range(len(cats)):
    for j in range(i+1, len(cats)):
        pairs.append((cats[i], cats[j], M[i, j]))
pairs_df = pd.DataFrame(pairs, columns=['category_a','category_b','cosine']).sort_values('cosine', ascending=False)
display(pairs_df.head(10).round(3))


In [ ]:
# @title Ячейка 16: Тепловая карта сходства классов
# @markdown Высокое сходство требует проверки реальной матрицей ошибок; не объявляем пары ошибочными заранее.

plt.figure(figsize=(10, 8))
sns.heatmap(pd.DataFrame(M, index=cats, columns=cats), annot=True, fmt='.2f',
            cmap='coolwarm', vmin=0, vmax=1, square=True)
plt.title('Косинусное сходство TF-IDF-центроидов категорий')
plt.tight_layout(); plt.show()


In [ ]:
# @title Ячейка 17: Флаги по категориям
# @markdown Описательная связь лингвистических flags и категорий. χ² + BH/FDR; не причинная связь.

from scipy.stats import chi2_contingency
flag_chars = list(sc.LINGUISTIC_FLAGS)
rows = []
for fc in flag_chars:
    has_flag = sc.flag_membership(df['flags'], fc)
    ct = pd.crosstab(df['category'], has_flag)
    if ct.shape[1] < 2:
        p, v = float('nan'), float('nan')
    else:
        chi2, p, dof, expected = chi2_contingency(ct)
        n = ct.values.sum()
        v = float(np.sqrt(chi2 / (n * (min(ct.shape) - 1)))) if min(ct.shape) > 1 else float('nan')
    rows.append({'flag': fc, 'meaning':sc.LINGUISTIC_FLAGS[fc],
                 'n_rows_with_flag':int(has_flag.sum()),
                 'share':float(has_flag.mean()),
                 'cramers_v':v,'p_value':p})
flag_assoc = pd.DataFrame(rows)
# Benjamini-Hochberg adjustment across documented flags only.
valid = flag_assoc['p_value'].notna()
p_values = flag_assoc.loc[valid, 'p_value'].to_numpy(float)
if len(p_values):
    order = np.argsort(p_values)
    sorted_p = p_values[order]
    adjusted_sorted = np.minimum.accumulate((sorted_p * len(p_values) /
                           np.arange(1, len(p_values) + 1))[::-1])[::-1]
    adjusted = np.empty_like(p_values)
    adjusted[order] = np.minimum(1.0, adjusted_sorted)
    flag_assoc.loc[valid, 'p_fdr_bh'] = adjusted
flag_assoc = flag_assoc.sort_values('cramers_v', ascending=False)
display(flag_assoc)
print('Связи с категориями описывают структуру Bitext и НЕ подтверждают причинное влияние флага на качество.')


### Выводы по этапам 2–3

EDA описывает **шаблонный англоязычный Bitext**, а не реальные обращения магазина. По [официальной карточке Bitext 2020](https://www.kaggle.com/datasets/bitext/training-dataset-for-chatbotsvirtual-assistants/metadata) подтверждены **12 кодов**: B (базовый синтаксис), S (синтаксис), L (лексическая вариативность), M (морфология), I (вопросы), C (сложные/сочинённые конструкции), P (вежливость), Q (разговорность), W (грубая лексика), E (развёрнутые сокращения), D (косвенная речь), Z (шум/опечатки/пунктуация).

Коды могут встречаться совместно. **Z — не автоматически OOD**, **W — не автоматически P1**, а C **не является разметкой нескольких независимых запросов**. Исследование флагов выявляет срезы ошибок, но не доказывает причинную устойчивость модели и не заменяет независимый тест на реальных данных.

**Этап 4. Предобработка и разбиение**

Очистка → char/word-n-gram группы перефразировок и опечаток → непересекающиеся `train/calibration/val/test`. Stratification **приближённая**, поскольку группы нельзя разрезать. Если обнаруживаются очень большие компоненты, код требует проверки. Текст test нельзя использовать для исправления сплита после просмотра метрик.

In [ ]:
# @title Ячейка 18: Очистка данных
# Промежуточная проверка; окончательный набор строит clean_frame в ячейке 20.
preliminary = raw.copy()
preliminary['normalized'] = preliminary.utterance.fillna('').str.lower().str.strip()
print('Дубликаты lower+strip:', preliminary.normalized.duplicated().sum())
print('Пустые/пропущенные:', preliminary.normalized.eq('').sum())
print('Конфликтов категорий:', (preliminary.groupby('normalized').category.nunique() > 1).sum())


In [ ]:
# @title Ячейка 19: Функция preprocess()
preprocess = sc.preprocess
examples_preprocess = ["I can't log in", 'I can log in', 'Invoice for {{Order Number}}',
                       'Invoice for {Order Number}', 'Invoice for <Order Number>', '  HELLO   there  ']
display(pd.DataFrame({'raw': examples_preprocess, 'normalized': list(map(preprocess, examples_preprocess))}))
assert all(preprocess(preprocess(t)) == preprocess(t) for t in examples_preprocess)
assert preprocess("I can't log in") != preprocess('I can log in')


In [ ]:
# @title Ячейка 20: Очистка и подготовка к сплиту
d, short, cleaning_info = sc.clean_frame(raw)
display(pd.Series(cleaning_info, name='cleaning'))
display(short[['utterance', 'category', 'intent', 'n_words']])
print('Короткие обращения сохранены. Flags одинаковых нормализованных текстов объединены.')


In [ ]:
# @title Ячейка 21: Групповое разбиение с балансировкой категорий
# До разбиения образуем char_wb 3-5 OR word 1-2 группы при порогах 0.85 / 0.90.
# Пороги фиксированы заранее, test используется только после финального выбора.
d['group'], group_info = sc.lexical_groups(d.text.tolist(),threshold=.85)
cluster_sizes=d.group.value_counts()
ambiguous_clusters=d.groupby('group').agg(categories=('category','nunique'),
    intents=('intent','nunique'),rows=('group','size')).query('categories > 1 or intents > 1')
cluster_examples=d[d.group.isin(ambiguous_clusters.index)].sort_values('group')
cluster_examples[['group','text','category','intent']].to_csv(REPORTS_DIR/'mixed_label_groups_review.csv',index=False)
assert cluster_sizes.max()/len(d) < .30, 'Крупная компонента: проверьте случаи переобъединения (mixed_label_groups_review.csv)'
print('Largest groups:',cluster_sizes.head(10).to_dict())
print('Cross-label clusters for manual audit:',len(ambiguous_clusters))
assert sc.lexical_groups(['I need to pay for my roder','I need to pay for my order'],.85)[0][0] == sc.lexical_groups(['I need to pay for my roder','I need to pay for my order'],.85)[0][1]
assert sc.lexical_groups(['problem whlie creating a profile','problem creating a profile'],.85)[0][0] == sc.lexical_groups(['problem whlie creating a profile','problem creating a profile'],.85)[0][1]
splits = sc.make_splits(d)
train,calibration,val,test=[splits[k] for k in ['train','calibration','val','test']]
display(pd.Series(group_info))
split_summary=pd.DataFrame([{'split':k,'n':len(v),'fraction':len(v)/len(d),
                             'groups':v.group.nunique(),'categories':v.category.nunique()} for k,v in splits.items()])
display(split_summary)
display(pd.DataFrame({k:v.category.value_counts() for k,v in splits.items()}).fillna(0).astype(int))
print('Важно: отсутствие общих компонентов не гарантирует отсутствия смысловых парафразов.')

In [ ]:
# @title Ячейка 22: Проверка на утечку (после дедупликации)
overlap_checks=sc.assert_split_integrity(splits)
display(pd.DataFrame(overlap_checks).T)
similarity_checks={}
for name in ['calibration','val','test']:
    summary,pairs=sc.cross_split_similarity(train,splits[name])
    similarity_checks[name]=summary
    pairs.to_csv(REPORTS_DIR/f'similar_train_{name}.csv',index=False)
    print(name,summary)
    display(pairs.head(8))
print('Два пространства диагностики — word и char, обучаются ТОЛЬКО на train.')
print('Если остаются пары ≥0.85, задокументируйте их; НЕ пересоздавайте split по результатам test.')

In [ ]:
# @title Ячейка 23: Сохранение сплитов
for name, frame in splits.items():
    frame.to_csv(PROCESSED_DIR/f'{name}.csv', index=False, lineterminator='\n')
short.to_csv(PROCESSED_DIR/'short_examples_retained.csv', index=False)
split_manifest = {'seed': SEED, 'raw_sha256': sc.sha256_file(raw_path),
                  'cleaning': cleaning_info, 'grouping': group_info,
                  'overlap_checks': overlap_checks, 'similarity_checks': similarity_checks,
                  'roles': {'train':'model selection and fitting', 'calibration':'sigmoid calibration only',
                            'val':'routing policy selection', 'test':'final in-corpus evaluation'},
                  'splits': {k:{'rows':len(v), 'sha256':sc.sha256_file(PROCESSED_DIR/f'{k}.csv'),
                                 'class_counts':v.category.value_counts().to_dict()} for k,v in splits.items()}}
sc.write_json(PROCESSED_DIR/'split_manifest.json', split_manifest)
print('Сохранены четыре CSV и проверяемый манифест.')


### 📝 Выводы по этапу 4

Группы строятся по `char_wb` 3–5 и word 1–2, чтобы опечатки вроде `roder/order` и `whlie/while` не проходили в разные выборки. Проверка высокой близости выполняется отдельным train-only пространством, в отчёте сохраняются оставшиеся пары ≥0.85. Группы — лексическая эвристика, а не гарантия отсутствия смысловых повторов.

# *Этап 5. Baseline*

Все признаки обучаются внутри Pipeline. Диагностики групп/интентов/flags выполняются на train. Validation ниже даёт описательные baseline-метрики, но выбор модели делается только по DEV-LOIO внутри train.


In [ ]:
# @title Ячейка 24: Загрузка сплитов из data/processed
splits = {k:pd.read_csv(PROCESSED_DIR/f'{k}.csv', keep_default_na=False) for k in ['train','calibration','val','test']}
train, calibration, val, test = [splits[k] for k in ['train','calibration','val','test']]
sc.assert_split_integrity(splits)
print({k: len(v) for k,v in splits.items()})


In [ ]:
# @title Ячейка 25: DummyClassifier — нижняя граница
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(train),1)), train.category)
dummy_pred = dummy.predict(np.zeros((len(val),1)))
dummy_metrics = sc.classification_metrics(val.category, dummy_pred)
print('Dummy на val:', {k:dummy_metrics[k] for k in ['accuracy','macro_f1','errors']})


In [ ]:
# @title Ячейка 26: Baseline — TF-IDF + LogisticRegression
BASELINE_NAME = 'lr_word (baseline)'
def make_lr_baseline(): return sc.make_model(BASELINE_NAME)
baseline = sc.fit_model(BASELINE_NAME, train)
base_val_metrics = sc.classification_metrics(val.category, baseline.predict(val.text.tolist()))
display(pd.DataFrame(base_val_metrics['report']).T)
print('Pipeline использует только текст; class_weight=balanced. Val не используется для fitting.')


In [ ]:
# @title Ячейка 27: Топ-признаки по классам
feature_names = baseline.named_steps['features'].get_feature_names_out()
clf = baseline.named_steps['clf']
top_features = []
for j, category in enumerate(clf.classes_):
    idx = np.argsort(clf.coef_[j])[-8:][::-1]
    top_features.append({'category': category, 'positive_terms': ', '.join(feature_names[idx])})
display(pd.DataFrame(top_features))
print('Коэффициенты линейной модели: связь с решением, не причинное объяснение.')


In [ ]:
# @title Ячейка 28: Кривая обучения (bias/variance)
from sklearn.model_selection import learning_curve, StratifiedGroupKFold
cv = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(train.text, train.category, train.group))
with threadpool_limits(limits=1):
    sizes, train_scores, cv_scores = learning_curve(make_lr_baseline(), train.text.tolist(), train.category,
        cv=cv, train_sizes=np.linspace(.1,.9,8), scoring='f1_macro', n_jobs=1, shuffle=True, random_state=SEED,
        error_score='raise')
plt.figure(figsize=(8,4))
plt.plot(sizes, train_scores.mean(1), 'o-', label='train')
plt.plot(sizes, cv_scores.mean(1), 'o-', label='group CV')
plt.xlabel('Число обучающих примеров'); plt.ylabel('macro-F1'); plt.legend(); plt.show()
print('Векторизатор заново обучается внутри каждого фолда; группы не пересекаются с валидацией.')


In [ ]:
# @title Ячейка 29: GroupSplit — разбиение по группам перефразировок
# Holdout целых групп, а не тест только на одиночках. Внешние val/test не участвуют.
tr_gs, te_gs = sc.balanced_group_holdout(train, .20, seed=99)
assert not set(tr_gs.group) & set(te_gs.group)
baseline_gs = sc.fit_model(BASELINE_NAME, tr_gs)
base_group_metrics = sc.classification_metrics(te_gs.category, baseline_gs.predict(te_gs.text.tolist()))
print('Group diagnostic:', {'train':len(tr_gs), 'heldout':len(te_gs), 'macro_f1':base_group_metrics['macro_f1']})


In [ ]:
# @title Ячейка 30: Проверка переноса между срезами по лингвистическим флагам
# @markdown Флаги могут пересекаться; категории и интенты не сбалансированы между подвыборками.
FLAG_MEANINGS = sc.LINGUISTIC_FLAGS  # Все 12 кодов из Bitext datacard, не только Z/Q/W/M/E/S.
def flag_diagnostic(frame, model_name):
    rows = []
    for flag, meaning in FLAG_MEANINGS.items():
        mask = sc.flag_membership(frame['flags'], flag)
        te = frame[mask]
        tr = frame[~mask & ~frame.group.isin(te.group)]
        absent_categories = sorted(set(te.category) - set(tr.category))
        row = {'model':model_name,'flag':flag,'meaning':meaning,
               'train':len(tr),'heldout':len(te),
               'heldout_share':len(te)/len(frame),
               'heldout_categories':int(te.category.nunique()),
               'missing_train_categories':absent_categories}
        if len(te) < 30 or tr.category.nunique() < 2 or absent_categories:
            row.update(status='INSUFFICIENT_DATA',macro_f1=None)
        else:
            m = sc.fit_model(model_name, tr)
            # Macro-F1 is defined over all categories in this held-out flagged slice.
            metrics = sc.classification_metrics(te.category,
                m.predict(te.text.tolist()), labels=sorted(te.category.unique()))
            row.update(status='OBSERVATIONAL_FLAG_HOLDOUT', macro_f1=metrics['macro_f1'])
        rows.append(row)
    return pd.DataFrame(rows)
flag_df = flag_diagnostic(train, BASELINE_NAME)
display(flag_df)
print('Срезы пересекаются, поэтому 12 строк нельзя считать независимыми экспериментами.')
print('Z=шум/опечатки, Q=разговорность, W=грубая речь, M=морфология, E=развёрнутые сокращения.')


In [ ]:
# @title Ячейка 31: Leave-one-intent-out на обучающей части
DEV_INTENTS, DIAG_INTENTS, loio_excluded = sc.intent_protocol(train)
print('DEV-интенты (без повторных seed):', DEV_INTENTS)
print('Оставшиеся диагностические интенты:', len(DIAG_INTENTS))
print('Категории с одним интентом: обычный LOIO невозможен без удаления всего класса.')
display(loio_excluded)
base_dev_rows, base_dev_preds = sc.loio_evaluate(train, BASELINE_NAME, DEV_INTENTS)
display(base_dev_rows)
print('Средний recall по DEV-интентам:', base_dev_rows.recall.mean())


In [ ]:
# @title Ячейка 32: Итоговая таблица baseline
baseline_overview = pd.DataFrame([
    {'protocol':'val', 'metric':'macro_f1', 'value':base_val_metrics['macro_f1']},
    {'protocol':'group holdout within train', 'metric':'macro_f1', 'value':base_group_metrics['macro_f1']},
    {'protocol':'DEV LOIO within train', 'metric':'mean_intent_recall', 'value':base_dev_rows.recall.mean()}])
display(baseline_overview)
print('Эти протоколы отвечают на разные вопросы; средний recall одного интента не называем macro-F1.')


### 📝 Выводы по этапу 5

Baseline и диагностики измеряются на разных срезах: `val` — внутридатасетная проверка, GroupSplit — новые лексические группы, LOIO — исключённый интент при сохранённом классе. Когда `payment_issue` составляет почти весь PAYMENT, оставшееся обучение «голодает»: recall в этом фолде нельзя приравнять к оценке настоящего нового интента. Исходный test остаётся закрыт до этапа 7.

# **Этап 6. Улучшение модели**

Сравниваются **11 названий / 10 различимых LOIO-моделей на всех 25 допустимых интентах**. Авторские 33 DEV и 8 обучающих фраз не получают никакого веса в ранжировании; `semantic_aug_lr` и `semantic_lr` идентичны в честном LOIO и это явно показывается. Нестабильность по интентам оценивается 2000 парными bootstrap-пересэмплированиями интентов. Разработка на 25 интентах не является независимым доказательством обобщения.

In [ ]:
# @title Ячейка 33: 11 конфигураций моделей
from functools import partial
CONFIGS={name:partial(sc.make_model,name) for name in sc.CONFIG_NAMES}
manual_dev,manual_holdout=manual_cases.frames()
manual_dev.to_csv(REPORTS_DIR/'manual_dev_cases.csv',index=False)
manual_holdout.to_csv(REPORTS_DIR/'manual_holdout_cases.csv',index=False)
added_training=pd.DataFrame(category_examples.EXAMPLES)
added_training.to_csv(PROCESSED_DIR/'supplemental_training_examples.csv',index=False)
all_authored=pd.concat([manual_dev,manual_holdout,added_training],ignore_index=True)
assert not all_authored.text.map(sc.preprocess).duplicated().any()
assert not set(all_authored.text.map(sc.preprocess)) & set(d.text)
print('Авторские 33 DEV + 33 HOLDOUT + 8 TRAIN не независимы по источнику/авторству.')
print('8 примеров отписки не включены в LOIO и НЕ могут решить выбор модели. Они также не используются финальным победителем по умолчанию.')
display(pd.DataFrame([{'model':name,'classifier':type(factory().named_steps['clf']).__name__,
    'features':type(factory().named_steps['features']).__name__} for name,factory in CONFIGS.items()]))

In [ ]:
# @title Ячейка 34: Сравнение 11 конфигураций по 25 исключаемым интентам
# Вариант с авторскими дополнениями использует те же прогнозы LOIO, что и semantic_lr_C1.
ALL_LOIO_INTENTS=DEV_INTENTS+DIAG_INTENTS
print('Число допустимых LOIO интентов:',len(ALL_LOIO_INTENTS))
tuning_cache={};tuning_rows=[]
for name in CONFIGS:
    started=time.monotonic()
    if name=='semantic_aug_lr_C1':
        rows,preds=tuning_cache['semantic_lr_C1']
        rows=rows.copy();preds=preds.copy();preds['model']=name
        distinguishable=False
    else:
        rows,preds=sc.loio_evaluate(train,name,ALL_LOIO_INTENTS)
        distinguishable=True
    tuning_cache[name]=(rows,preds)
    assert rows.intent.nunique()==len(ALL_LOIO_INTENTS)
    mean=float(rows.recall.mean())
    tuning_rows.append({'model':name,'mean_intent_recall':mean,
         'std_between_intents':float(rows.recall.std()),
         'standard_error_exploratory':float(rows.recall.std()/np.sqrt(rows.recall.notna().sum())),
         'n_intents':int(rows.recall.notna().sum()),'class_starvation_folds':int(rows.class_starvation_warning.sum()),
         'loio_indistinguishable_from_semantic_lr':not distinguishable,
         'selection_eligible':distinguishable,'seconds':time.monotonic()-started})
    print(name, 'LOIO',round(mean,4),'folds',len(rows),
          'starved',int(rows.class_starvation_warning.sum()),flush=True)
    # Encoder caching, if used, is for frozen pre-trained features only; no test fitting.
    if name.startswith('semantic_'):
        semantic_encoder.save_cache()
tune_df=pd.DataFrame(tuning_rows).sort_values('mean_intent_recall',ascending=False).reset_index(drop=True)
assert set(tune_df.model) == set(sc.CONFIG_NAMES) and tune_df.model.is_unique
assert tune_df.n_intents.eq(len(ALL_LOIO_INTENTS)).all()
tune_df.to_csv(REPORTS_DIR/'loio_all_configurations.csv',index=False)
pd.concat([v[0].assign(model=k) for k,v in tuning_cache.items()]).to_csv(
   REPORTS_DIR/'loio_all_models_per_intent.csv',index=False)
pd.concat([v[1].assign(model=k) for k,v in tuning_cache.items()]).to_csv(
   REPORTS_DIR/'loio_all_models_predictions.csv',index=False)
display(tune_df)
print('ВЫБОР пока не выполнен. Авторский DEV не получает вес 50%.')

In [ ]:
# @title Ячейка 35: Правило выбора модели
# Reference задан до оценки: char+word LinearSVC. Полная таблица LOIO участвует в разработке,
# т.е. bootstrap-интервалы НЕ являются независимой проверкой победителя.
SELECTION_REFERENCE='linsvc_wc_C1'
loio_all_models=pd.concat([v[0].assign(model=k) for k,v in tuning_cache.items()],ignore_index=True)
selection_bootstrap=[sc.paired_intent_bootstrap(loio_all_models,SELECTION_REFERENCE,name,iterations=2000)
     for name in CONFIGS if name!=SELECTION_REFERENCE]
selection_ci_df=pd.DataFrame(selection_bootstrap)
selection_ci_df['ci_lower']=selection_ci_df.paired_delta_ci95.map(lambda z:z[0])
selection_ci_df['ci_upper']=selection_ci_df.paired_delta_ci95.map(lambda z:z[1])
selection_ci_df.to_csv(REPORTS_DIR/'loio_paired_bootstrap.csv',index=False)
observed=tune_df[tune_df.selection_eligible].iloc[0]
if observed.model==SELECTION_REFERENCE:
    best_model_name=SELECTION_REFERENCE
    selection_reason='Reference has highest observable mean on eligible candidates'
else:
    ci=selection_ci_df.set_index('candidate').loc[observed.model]
    if ci.ci_lower>0:
        best_model_name=observed.model
        selection_reason='Exploratory paired-intent CI lower bound above fixed reference'
    else:
        best_model_name=SELECTION_REFERENCE
        selection_reason='Observed difference vs fixed reference overlaps zero; prefer parsimonious reference'
# Distinguishability rule: no author-written extension can win on identical LOIO predictions.
assert best_model_name!='semantic_aug_lr_C1'
selection_record={'selected':best_model_name,'reference':SELECTION_REFERENCE,
   'observed_leader':observed.model,'reason':selection_reason,
   'manual_dev_selection_weight':0,'validation_roles':'25 LOIO intents for development; val for shadow policy; test untouched',
   'not_independent':'Selection and bootstrap both use same 25 intent folds'}
display(selection_ci_df.sort_values('paired_mean_delta',ascending=False))
display(pd.Series(selection_record))
print('Нет выводов о реальном потоке из этой процедуры: 25 интентов внутри Bitext.')

In [ ]:
# @title Ячейка 36: Диагностика на обучающей части по интентам, группам и флагам
# Эти 25 интентов участвуют в выборе модели.
# DEV-10 / remaining-15 сохраняются для воспроизведения ревью, но НЕ являются отложенным тестом.
loio_all={k:v[0] for k,v in tuning_cache.items()}
loio_predictions={k:v[1] for k,v in tuning_cache.items()}
loio_summary=[]
for name, rows in loio_all.items():
    preds=loio_predictions[name]
    metric=sc.classification_metrics(preds.y_true,preds.y_pred,labels=sorted(preds.y_true.unique()))
    dev=rows[rows.intent.isin(DEV_INTENTS)]
    rest=rows[rows.intent.isin(DIAG_INTENTS)]
    loio_summary.append({'model':name,'dev10_mean_recall':float(dev.recall.mean()),
       'remaining15_mean_recall':float(rest.recall.mean()),
       'all25_mean_recall':float(rows.recall.mean()),
       'aggregate_macro_f1_eligible_categories':metric['macro_f1'],
       'n_intents':len(rows),'class_starvation_folds':int(rows.class_starvation_warning.sum()),
       'selection_eligible':name!='semantic_aug_lr_C1'})
loio_summary_df=pd.DataFrame(loio_summary).sort_values('all25_mean_recall',ascending=False)
display(loio_summary_df)
loio_summary_df.to_csv(REPORTS_DIR/'loio_comparison_all_11.csv',index=False)
loio_excluded.to_csv(REPORTS_DIR/'loio_single_intent_categories.csv',index=False)
comparison_names=list(dict.fromkeys([BASELINE_NAME,'linsvc_wc_C1',best_model_name]))
group_rows=[]
for name in comparison_names:
    m=sc.fit_model(name,tr_gs)
    group_rows.append({'model':name,**{k:v for k,v in sc.classification_metrics(
       te_gs.category,m.predict(te_gs.text.tolist())).items() if k in ['macro_f1','accuracy','n']}})
group_df=pd.DataFrame(group_rows)
display(group_df)
best_flag_df=flag_diagnostic(train,best_model_name)
display(pd.concat([flag_df,best_flag_df],ignore_index=True))
# Авторские DEV (33) теперь только post-selection диагностика, не критерий.
authored_dev_model=sc.fit_calibrated_bundle(best_model_name,train,calibration)
manual_dev_pred=authored_dev_model.predict(manual_dev.text.tolist())
manual_dev_metrics=sc.classification_metrics(manual_dev.category,manual_dev_pred)
pd.DataFrame({'text':manual_dev.text,'category':manual_dev.category,
              'prediction':manual_dev_pred}).to_csv(REPORTS_DIR/'manual_dev_predictions_selected.csv',index=False)
print('Авторские DEV: верных',len(manual_dev)-manual_dev_metrics['errors'],'/',len(manual_dev),
      'Wilson lower',sc.wilson_lower(len(manual_dev)-manual_dev_metrics['errors'],len(manual_dev)))
print('Это самодельный набор того же автора, НЕЛЬЗЯ считать внешней валидацией.')

# **Этап 7. Финальная оценка, уверенность, маршрутизация**

Обучение — только `train`, calibration — только `calibration`, выбор *shadow* порогов — `val + авторский OOD-dev`; test используется ровно для финального контроля. **Операционная автоматика выключена по умолчанию**: нет 300–500 независимых размеченных реальных ID, ≥200 реальных OOD, P1-метрик и утверждённых потерь ошибок. Shadow — диагностическая симуляция, а не одобрение очереди.

In [ ]:
# @title Ячейка 37: Обучение на train, калибровка и выбор теневых порогов
# Калибровка использует отдельную часть. Пороги выбираются на val и синтетическом OOD-dev.
ood_dev,ood_test=scenario_data.frames()
assert not set(ood_dev.text.map(sc.preprocess)) & set(ood_test.text.map(sc.preprocess))
assert not set(pd.concat([ood_dev,ood_test]).text.map(sc.preprocess)) & set(d.text)
final_bundle=sc.fit_calibrated_bundle(best_model_name,train,calibration)
baseline_final=sc.fit_calibrated_bundle(BASELINE_NAME,train,calibration)
policy,policy_table=sc.select_policy(final_bundle,val,ood_dev,
    target_error_upper=.01,min_auto=30,false_auto_cost=25.,review_cost=1.)
from support_router.cascade import fit_business_cascade
fit_business_cascade(final_bundle, train, calibration, val, ood_dev)
assert policy['auto_enabled'] is False
shadow_model=sc.shadow_bundle(final_bundle)
final_bundle.metadata={'split_manifest':split_manifest,'model_selection':selection_record,
   'fit_rows':len(train),'calibration_rows':len(calibration),'policy_rows':len(val),
   'synthetic_ood_dev_rows':len(ood_dev),'training_extension':[],
   'manual_dev_n':len(manual_dev),'manual_holdout_n':len(manual_holdout),
   'operational_auto_enabled':False,'real_external_validation':False,
   'feature_columns':['text'],'excluded_feature_columns':['intent','flags','group','row_id','category'],
   'encoder_revision':semantic_encoder.REVISION if best_model_name.startswith('semantic_') else None}
calibration_val=sc.calibration_diagnostics(final_bundle,val)
final_model=final_router_model=final_bundle
print('Финальная модель:',final_bundle.model_name,final_bundle.model_id)
display(pd.Series(policy))
manual_holdout_routes=sc.batch_route(final_bundle,manual_holdout.text.tolist())
manual_holdout_shadow=sc.batch_route(shadow_model,manual_holdout.text.tolist())
manual_holdout_pred=final_bundle.predict(manual_holdout.text.tolist())
manual_holdout_metrics=sc.classification_metrics(manual_holdout.category,manual_holdout_pred)
manual_holdout_routing=sc.routing_metrics(manual_holdout_routes,manual_holdout.category.to_numpy())
manual_holdout_shadow_routing=sc.routing_metrics(manual_holdout_shadow,manual_holdout.category.to_numpy())
manual_holdout_report=manual_holdout.assign(prediction=manual_holdout_pred,
    deployed_status=[r['status'] for r in manual_holdout_routes],
    shadow_status=[r['status'] for r in manual_holdout_shadow],
    shadow_confidence=[r['confidence'] for r in manual_holdout_shadow],
    shadow_similarity=[r['similarity'] for r in manual_holdout_shadow],
    reasons=[r['reason_codes'] for r in manual_holdout_shadow])
manual_holdout_report.to_csv(REPORTS_DIR/'manual_holdout_predictions.csv',index=False)
display(manual_holdout_report)
print('Авторский HOLDOUT:',len(manual_holdout)-manual_holdout_metrics['errors'],'/',len(manual_holdout),
      'category Wilson lower:',sc.wilson_lower(len(manual_holdout)-manual_holdout_metrics['errors'],len(manual_holdout)))
print('SHADOW coverage',manual_holdout_shadow_routing['coverage'],
      '| реально auto_disabled всегда. Не меняем модель/пороги после просмотра.')

In [ ]:
# @title Ячейка 38: ЕДИНСТВЕННАЯ оценка на test + McNemar
# Test на Bitext — регрессионный контроль шаблонного корпуса, НЕ независимый реальный поток.
y_test=test.category.to_numpy()
pred_base=baseline_final.predict(test.text.tolist())
pred_best=final_bundle.predict(test.text.tolist())
base_test_metrics=sc.classification_metrics(y_test,pred_base)
test_metrics=sc.classification_metrics(y_test,pred_best)
mcnemar_result=sc.paired_error_test(y_test,pred_base,pred_best)
test_routes=sc.batch_route(final_bundle,test.text.tolist())
test_routing=sc.routing_metrics(test_routes,y_test)
test_shadow_routes=sc.batch_route(shadow_model,test.text.tolist())
test_shadow_routing=sc.routing_metrics(test_shadow_routes,y_test)
display(pd.DataFrame(test_metrics['report']).T)
print('Baseline:',{k:base_test_metrics[k] for k in ['macro_f1','accuracy','errors']})
print('Selected:',{k:test_metrics[k] for k in ['macro_f1','accuracy','errors']})
print('McNemar:',mcnemar_result)
print('Реальная (не включенная) автоматика:',test_routing)
print('Только SHADOW-кандидат (category accuracy):',test_shadow_routing)
print('Насыщенный test не различает модели надёжно. Пояснение: переобученные шаблоны и близкие парафразы.')
test_predictions=test[['row_id','group','utterance','category','intent']].copy()
test_predictions['baseline_prediction']=pred_base
test_predictions['model_prediction']=pred_best
for field in ['category','decision_category','accepted_category','queue','confidence','similarity','status','priority','reason_codes']:
    test_predictions['route_'+field]=[r[field] for r in test_shadow_routes]
test_predictions['deployed_status']=[r['status'] for r in test_routes]
test_predictions.to_csv(REPORTS_DIR/'test_predictions.csv',index=False)
short_test_mask=test.n_words.lt(3).to_numpy()
short_test_metrics={'n':int(short_test_mask.sum()),
    'classification_accuracy':float(np.mean(pred_best[short_test_mask]==y_test[short_test_mask])) if short_test_mask.any() else None,
    'shadow_routing':sc.routing_metrics([r for r,ok in zip(test_shadow_routes,short_test_mask) if ok],
                                  y_test[short_test_mask]) if short_test_mask.any() else None,
    'limitation':'Small corpus slice, not evidence of robust short real customer tickets'}
print('Короткие тексты:',short_test_metrics)

In [ ]:
# @title Ячейка 39: Bootstrap CI95 для macro-F1
bootstrap_result=sc.paired_group_bootstrap(test,pred_base,pred_best,iterations=2000)
display(pd.Series(bootstrap_result))
print('2000 paired cluster resamples. Это лишь вариация внутри шаблонных лексических групп Bitext.')

In [ ]:
# @title Ячейка 40: Проверка калибровки на val
# Калибратор уже обучен в ячейке 37. Здесь только диагностика той же модели на val.
display(pd.Series({k:v for k,v in calibration_val.items() if k!='bins'}))
reliability = pd.DataFrame(calibration_val['bins'])
display(reliability)
plt.figure(figsize=(6,5)); plt.plot([0,1],[0,1],'--',color='gray')
plt.plot(reliability.confidence, reliability.accuracy, 'o-')
plt.xlabel('Средняя уверенность'); plt.ylabel('Доля верных решений'); plt.title('Reliability на val'); plt.show()
if calibration_val['confidence_auroc'] is None:
    print('AUROC не определён: correctness имеет один класс. Это не означает слабость score.')
print('Достаточность вероятностей на реальном потоке не подтверждена этим шаблонным корпусом.')


In [ ]:
# @title Ячейка 41: Порог уверенности для маршрутизации (исправлено)
# Порог кандидата заморожен ДО test; операции AUTO остаются отключены.
display(pd.Series(policy))
display(policy_table[policy_table.eligible].head(15))
policy_table.to_csv(REPORTS_DIR/'policy_validation_grid.csv',index=False)
if policy['candidate_auto_enabled']:
    part=policy_table[policy_table.similarity_threshold.eq(policy['similarity_threshold'])]
    plt.figure(figsize=(7,4))
    plt.plot(part.coverage,part.category_error_upper_wilson_95,'o-')
    plt.axhline(policy['target_error_upper'],linestyle='--',color='red')
    plt.xlabel('Покрытие val (кандидат)')
    plt.ylabel('Wilson upper для ошибок категории')
    plt.show()
else:
    print('Ни один shadow-кандидат не прошёл демонстрационный критерий.')
print('НЕ гарантия ≤1%: интервал вычислен на selection data после перебора сетки, реальные OOD не участвовали.')

In [ ]:
# @title Ячейка 42: Синтетический OOD-набор
# Сценарии написаны вручную, не являются данными реальных клиентов.
display(pd.concat([ood_dev, ood_test]).groupby(['role','topic']).size().unstack(fill_value=0))
print('OOD-dev участвует только в выборе политики, OOD-holdout — отдельная синтетическая диагностика.')
print('Исторические 18 OOD сохраняются как регрессионные примеры; модель на них не обучается.')


In [ ]:
# @title Ячейка 43: Решение OTHER и проверка на OOD
# OTHER означает воздержание, не классификационный класс.
# Отдельно показываем отключённую операционную маршрутизацию и SHADOW false-auto.
ood_holdout_routes=sc.batch_route(final_bundle,ood_test.text.tolist())
ood_holdout_shadow=sc.batch_route(shadow_model,ood_test.text.tolist())
ood_holdout_metrics=sc.routing_metrics(ood_holdout_routes)
ood_shadow_metrics=sc.routing_metrics(ood_holdout_shadow)
ood_shadow_metrics['false_auto_n']=ood_shadow_metrics['automatic_n']
ood_shadow_metrics['false_auto_rate']=ood_shadow_metrics['coverage']
ood_shadow_metrics['wilson_upper_95']=sc.wilson_upper(ood_shadow_metrics['false_auto_n'],len(ood_test))
ood_report=ood_test.copy()
for field in ['suggested_category','confidence','similarity','status','reason_codes']:
    ood_report['shadow_'+field]=[r[field] for r in ood_holdout_shadow]
ood_report['deployed_status']=[r['status'] for r in ood_holdout_routes]
ood_report['false_auto_shadow']=ood_report.shadow_status.eq('AUTO_RECOMMENDED')
display(ood_report)
display(ood_report.groupby('topic').false_auto_shadow.agg(['sum','count','mean']))
legacy_ood_routes=sc.batch_route(final_bundle,scenario_data.LEGACY_OOD)
legacy_ood_shadow=sc.batch_route(shadow_model,scenario_data.LEGACY_OOD)
legacy_ood_metrics=sc.routing_metrics(legacy_ood_routes)
legacy_ood_shadow_metrics=sc.routing_metrics(legacy_ood_shadow)
legacy_ood_shadow_metrics['false_auto_n']=legacy_ood_shadow_metrics['automatic_n']
legacy_ood_shadow_metrics['wilson_upper_95']=sc.wilson_upper(
    legacy_ood_shadow_metrics['automatic_n'],len(scenario_data.LEGACY_OOD))
print('Деплой (auto отключён):',ood_holdout_metrics)
print('Только SHADOW 24 фразы:',ood_shadow_metrics)
print('Только SHADOW legacy 18:',legacy_ood_shadow_metrics)
print('В ревью v2.1 обнаружено 2/24 false-auto на OOD-holdout и 3/18 на legacy; это ИСТОРИЧЕСКИЕ числа, не результат нового прогона.')
print('Regex-сценарии технических проблем пересекаются с авторским набором и не доказывают независимую полноту детектора.')
ood_report.to_csv(REPORTS_DIR/'ood_diagnostic.csv',index=False)

In [ ]:
# @title Ячейка 44: Демо-фразы (заранее зафиксированы)
demo_texts=scenario_data.DEMO
demo_routes=sc.batch_route(final_bundle,[t for t,_ in demo_texts])
demo_shadow=sc.batch_route(shadow_model,[t for t,_ in demo_texts])
demo_df=pd.DataFrame([{'text':t,'expected':expected,
   'suggested_category':r['suggested_category'],'decision_category':r['decision_category'],
   'queue':r['queue'],'confidence':r['confidence'],'status':r['status'],
   'shadow_status':sr['status'],'shadow_reasons':sr['reason_codes']}
   for (t,expected),r,sr in zip(demo_texts,demo_routes,demo_shadow)])
id_mask=demo_df.expected.ne('OTHER')
shadow_id=id_mask&demo_df.shadow_status.eq('AUTO_RECOMMENDED')
demo_metrics={'n':len(demo_df),'id_n':int(id_mask.sum()),
   'id_classification_correct':int((demo_df.loc[id_mask,'suggested_category']==demo_df.loc[id_mask,'expected']).sum()),
   'id_shadow_auto_n':int(shadow_id.sum()),'id_shadow_manual_n':int((id_mask&~shadow_id).sum()),
   'ood_n':int((~id_mask).sum()),
   'ood_shadow_manual_n':int((~id_mask & demo_df.shadow_status.ne('AUTO_RECOMMENDED')).sum())}
display(demo_df)
display(pd.Series(demo_metrics))
explanations=[sc.local_explanation(final_bundle,t) for t,_ in demo_texts[:3]]
display(pd.DataFrame(explanations))
demo_df.to_csv(REPORTS_DIR/'historical_demo.csv',index=False)
print('Ничего не подбирается по историческому демо. Это не независимая выборка.')

In [ ]:
# @title Ячейка 45: Сохранение модели и метаданных (исправлено)
artifact_path,artifact_meta=sc.save_bundle(final_bundle,MODELS_DIR,
    test_metrics={'classification':test_metrics,'shadow_routing':test_shadow_routing,
                  'operational_routing':test_routing})
semantic_encoder.save_cache()
# Fresh Python process: explicitly trusted LOCAL artifact created by this notebook.
probe_texts=[t for t,_ in demo_texts]+['I will hurt you','','Помогите с заказом']
sc.write_json(REPORTS_DIR/'parity_inputs.json',probe_texts)
probe_script="""
import json,sys
from pathlib import Path
import support_router as sc
root=Path(sys.argv[1])
m,meta=sc.load_bundle(root/'models',trust_local_artifact=True)
texts=json.loads((root/'reports/parity_inputs.json').read_text(encoding='utf-8'))
out={'probabilities':m.predict_proba(texts).tolist(),'routes':[sc.route_ticket(t,m) for t in texts]}
sc.write_json(root/'reports/fresh_process_predictions.json',out)
"""
env=dict(os.environ,PYTHONPATH=os.pathsep.join([str(PROJECT_DIR/'src'),str(PROJECT_DIR)]))
probe=subprocess.run([sys.executable,'-c',probe_script,str(PROJECT_DIR)],
               cwd=PROJECT_DIR,env=env,capture_output=True,text=True,encoding='utf-8',errors='replace')
if probe.returncode:
    raise RuntimeError('Проверка в отдельном процессе упала:\n'+probe.stderr[-3000:])
fresh=json.loads((REPORTS_DIR/'fresh_process_predictions.json').read_text(encoding='utf-8'))
assert np.allclose(final_bundle.predict_proba(probe_texts),fresh['probabilities'],rtol=0,atol=1e-12)
for text,exported in zip(probe_texts,fresh['routes']):
    live=sc.route_ticket(text,final_bundle)
    for field in ['category','decision_category','accepted_category','queue','status',
                  'confidence','reason_codes','priority','model_id','policy_version']:
        assert live[field]==exported[field],(text,field)
export_check={'fresh_process_parity':True,'n_cases':len(probe_texts),
              'probability_atol':1e-12,'artifact_sha256':artifact_meta['artifact_sha256'],
              'model_id':final_bundle.model_id,'trusted_local_only':True}
sc.write_json(REPORTS_DIR/'export_check.json',export_check)
display(pd.Series(export_check))
print('ВНИМАНИЕ: joblib.load использует pickle; meta SHA рядом с моделью НЕ аутентифицирует источник.')
print('Загружайте только свои локально обученные артефакты. vectorizer включён в RouterBundle.')

In [ ]:
# @title Ячейка 46: Сборка результатов и проверка доступности внешней разметки
import xml.etree.ElementTree as ET
# Регрессионные тесты проверяют корректность кода, не качество на живых обращениях.
test_env=dict(os.environ,PYTHONPATH=str(PROJECT_DIR),PYTEST_DISABLE_PLUGIN_AUTOLOAD='1')
test_run=subprocess.run([sys.executable,'-m','pytest','tests','-q',
                         '--junitxml=reports/tests.xml'],cwd=PROJECT_DIR,
                         env=test_env,capture_output=True,text=True,encoding='utf-8',errors='replace')
print(test_run.stdout)
if test_run.returncode:
    print(test_run.stderr)
    raise RuntimeError('Регрессионные тесты не прошли: не создаём ложный успешный results.json')
xml=ET.parse(REPORTS_DIR/'tests.xml').getroot()
suites=[e for e in xml.iter('testsuite')]
regression_tests={k:sum(int(x.attrib.get(k,0)) for x in suites)
                  for k in ['tests','failures','errors','skipped']}
# Внешние файлы, если пользователь предоставил человечески размеченную проверку.
# Никогда не используем эти данные для fit, выбора модели, порогов или повторной сдачи test.
external_validation={'id':{'status':'NOT_PROVIDED','min_required':300},
                     'ood':{'status':'NOT_PROVIDED','min_required':200},
                     'risk_p1':{'status':'NOT_PROVIDED','required':'human labels and recall'}}
real_id_path=PROCESSED_DIR/'real_blind_id.csv'  # columns utterance,category (+ provenance outside file)
real_ood_path=PROCESSED_DIR/'real_blind_ood.csv' # columns utterance
# Real examples must be independent and collected before viewing new results.
if real_id_path.exists():
    real_id=pd.read_csv(real_id_path).fillna('')
    assert {'utterance','category'}.issubset(real_id.columns)
    assert set(real_id.category).issubset(set(sc.CATEGORIES))
    assert len(real_id)>=300 and real_id.utterance.map(sc.preprocess).nunique()==len(real_id)
    assert not set(real_id.utterance.map(sc.text_id)) & set(d.row_id)
    real_id_predictions=final_bundle.predict(real_id.utterance.tolist())
    real_id_shadow=sc.batch_route(shadow_model,real_id.utterance.tolist())
    external_validation['id']={'status':'EVALUATED_USER_SUPPLIED_PROVENANCE_NOT_AUDITED',
       'n':len(real_id),'file_sha256':sc.sha256_file(real_id_path),
       'classification':sc.classification_metrics(real_id.category,real_id_predictions),
       'shadow_routing':sc.routing_metrics(real_id_shadow,real_id.category.to_numpy())}
if real_ood_path.exists():
    real_ood=pd.read_csv(real_ood_path).fillna('')
    assert 'utterance' in real_ood.columns and len(real_ood)>=200
    assert real_ood.utterance.map(sc.preprocess).nunique()==len(real_ood)
    assert not set(real_ood.utterance.map(sc.text_id)) & set(d.row_id)
    real_ood_shadow=sc.batch_route(shadow_model,real_ood.utterance.tolist())
    n_false=sum(r['status']=='AUTO_RECOMMENDED' for r in real_ood_shadow)
    external_validation['ood']={'status':'EVALUATED_USER_SUPPLIED_OOD_LABELS_NOT_AUDITED',
       'n':len(real_ood),'file_sha256':sc.sha256_file(real_ood_path),
       'shadow_false_auto_n':n_false,'shadow_false_auto_rate':n_false/len(real_ood),
       'wilson_upper_95':sc.wilson_upper(n_false,len(real_ood))}
real_risk_path=PROCESSED_DIR/'real_blind_risk.csv'  # columns utterance,risk_label (0/1 expert annotation)
if real_risk_path.exists():
    real_risk=pd.read_csv(real_risk_path).fillna('')
    assert {'utterance','risk_label'}.issubset(real_risk.columns)
    labels=pd.to_numeric(real_risk.risk_label,errors='raise').to_numpy()
    assert set(labels).issubset({0,1}) and len(real_risk)>=100
    assert not set(real_risk.utterance.map(sc.text_id)) & set(d.row_id)
    # Independently annotated risk labels, NOT generated by the same heuristic.
    detections=[sc.detect_signals(t) for t in real_risk.utterance]
    external_validation['risk_p1']={'status':'EVALUATED_USER_SUPPLIED_LABELS_NOT_AUDITED',
        'n':len(real_risk),'file_sha256':sc.sha256_file(real_risk_path),
        'recall_and_precision':sc.risk_recall(detections,labels),
        'note':'Risk labels need expert verification; recall CI reported by positive count'}
assert set(loio_summary_df.model) == set(sc.CONFIG_NAMES)
assert set(selection_ci_df.candidate) == set(sc.CONFIG_NAMES) - {SELECTION_REFERENCE}
results={'model_id':final_bundle.model_id,'model_name':best_model_name,
   'data':split_manifest,'selection':selection_record,
   'loio_11x25':tune_df.to_dict('records'),
   'loio_paired_intent_bootstrap':selection_bootstrap,
   'loio_summary':loio_summary_df.to_dict('records'),
   'group_diagnostics':group_df.to_dict('records'),
   'flag_diagnostics':pd.concat([flag_df,best_flag_df]).to_dict('records'),
   'baseline_test':base_test_metrics,'test':test_metrics,
   'test_operational_routing':test_routing,'test_shadow_routing':test_shadow_routing,
   'short_test_slice':short_test_metrics,'mcnemar':mcnemar_result,
   'bootstrap':bootstrap_result,'calibration_val':calibration_val,
   'policy':policy,'ood_holdout_synthetic_deployed':ood_holdout_metrics,
   'ood_holdout_synthetic_shadow':ood_shadow_metrics,
   'ood_legacy_deployed':legacy_ood_metrics,'ood_legacy_shadow':legacy_ood_shadow_metrics,
   'historical_demo':demo_metrics,'export_check':export_check,
   'regression_tests':regression_tests,
   'manual_dev_diagnostic':manual_dev_metrics,
   'manual_holdout':manual_holdout_metrics,
   'manual_holdout_deployed_routing':manual_holdout_routing,
   'manual_holdout_shadow_routing':manual_holdout_shadow_routing,
   'training_extension':final_bundle.metadata['training_extension'],
   'versions':sc.runtime_versions(),'stage8_9':{'status':'PENDING'},
   'business_policy_approved':False,'real_sla_data':None,
   'external_blind_validation':external_validation,
   'review_v2_1_historical_observations':{
       'test_baseline_macro_f1':.9982,'test_final_macro_f1':.9962,
       'mcnemar_p':.39,'ood_holdout_false_auto':'2/24',
       'legacy_ood_false_auto':'3/18',
       'note':'Historical reviewer values, NOT current rerun results'},
   'limitations':['Bitext is template-rich and not production representative',
       'No independent production labelled sample supplied by default',
       'Author-written diagnostics are not blind or statistically independent',
       'No validated P1 risk detector or business-approved routing thresholds',
       'Shadow policy does not send tickets; operational automation disabled']}
sc.write_json(REPORTS_DIR/'results.json',results)
print('Saved reports/results.json. Historical reviewer numbers kept separate from newly measured results.')
print('For production use, independent labelled ID/OOD/risk samples and business sign-off are mandatory.')

# 📝 Выводы по этапу 7

Ключевые факты ревью версии 2.1 (исторические, не результаты исправленной версии): **test насыщен** — baseline macro-F1 0.9982 (6 ошибок) против 0.9962 (10 ошибок) финальной модели, McNemar p≈0.39; выбор по DEV-10 не подтвердился на остальных 15 LOIO; на синтетическом OOD-holdout было **2/24 false-auto**, на legacy **3/18**. Высокая близость перефразировок между test и train делает такой test только регрессионной проверкой.

В версии 3.1 все 25 LOIO-фолдов участвуют в описанной заранее процедуре; обучение synthetic NEWSLETTER не может само «победить» по авторским примерам. Используется раздельная категория и очередь, включая ошибки категории внутри объединённой очереди. Shadow-пороги остаются гипотезой, эксплуатационный `auto_enabled=False`.

Эти меры **не создают отсутствующую реальную оценку**. Чтобы делать вывод о применимости, требуется внешняя человеческая разметка, включая P1-риски, и бизнес-согласование очередей и стоимости false-auto.

Лингвистические флаги интерпретируются строго по карточке **Bitext 2020**. Диагностика `flag_diagnostics` содержит 12 потенциально пересекающихся срезов и не называется контролируемым экспериментом «чистое → шумное»: разные составы категорий/интентов могут объяснить часть различий.


#***Этап 8-9 Маршрутизация обращений и веб-интерфейс***

Интерфейс загружает проверенный артефакт. Решение — рекомендация очереди и приоритета; интеграции с тикет-системой нет. История хранится отдельно в каждой сессии, содержит маскированные тексты и ограничена 100 записями.


In [ ]:
# @title Инициализация окружения и загрузка модели
model, loaded_meta = sc.load_bundle(MODELS_DIR,trust_local_artifact=True)
print('Модель:', loaded_meta['model_name'], loaded_meta['model_id'])
print('Классы:', loaded_meta['classes'])
print('Порог:', loaded_meta['policy']['confidence_threshold'], '| similarity:', loaded_meta['policy']['similarity_threshold'])
print('Локально созданный артефакт доверенный. SHA256 рядом с файлом НЕ защита от подмены; joblib небезопасен из чужих источников.')


In [ ]:
# @title Маршрутизация обращений и логика низкой уверенности
def route_ticket(text, context=None):
    return sc.route_ticket(text, model, context=context)
smoke_texts = ['I want to cancel my order', "I can't log in", 'I will hurt you',
               'I will not hurt you', 'You idiots, where is my refund?',
               'There is an unauthorized payment on my card', 'I will sue you', '', 'qzxv qqqq']
display(pd.DataFrame([{'text':t, **route_ticket(t)} for t in smoke_texts]))
# Пример серверного CRM-адаптера. Веб-пользователь не задаёт эти поля и не становится VIP по тексту.
mock_crm = {'source':'trusted_crm', 'vip_verified':True, 'open_related_ticket':True, 'contact_count':3}
print('Только демонстрация CRM:', route_ticket('Where is my refund?', context=mock_crm))
from datetime import datetime, timezone, timedelta
now = datetime.now(timezone.utc)
demo_tickets = [{'id':'ordinary-old','priority':'P3','created_at_utc':(now-timedelta(minutes=180)).isoformat()},
                {'id':'urgent','priority':'P1','created_at_utc':(now-timedelta(minutes=10)).isoformat()},
                {'id':'new-vip','priority':'P2','created_at_utc':now.isoformat()}]
scheduling_demo = sc.schedule_demo(demo_tickets, now)
display(pd.DataFrame(scheduling_demo['ordered']))
print('SLA-метрики отсутствуют. Aging/резерв/лимит — параметры синтетической демонстрации:', scheduling_demo['parameters'])


In [ ]:
# @title Веб-интерфейс (Дашборд) с Top-3, логированием и примерами
import gradio as gr
LAUNCH_UI = os.environ.get('SUPPORT_NONINTERACTIVE') != '1'   # True: открыть интерфейс после полного обучения или загрузки готовых моделей.
SHARE_DEMO = True  # True создаёт доступную по ссылке демонстрацию; используйте только обезличенные тексты.
HISTORY_COLUMNS = ['Время UTC','Текст (маскир.)','Категория','Очередь','Приоритет','Статус','Причины','model_id','policy_version']
from app import deployed_message

def process_web_request(user_text, history, detect_multiple=False):
    result, updated = sc.web_request(user_text, history, model, detect_multiple=detect_multiple)
    message = deployed_message(result)
    return (result['category'] or 'Не определена', result['queue'] or '', result['status'], result['priority'],
            '; '.join(result['reason_codes']), result['top3'],
            pd.DataFrame(updated[::-1], columns=HISTORY_COLUMNS), updated, message)
with gr.Blocks(title='Support Ticket Router', analytics_enabled=False) as demo:
    gr.Markdown('# Маршрутизация обращений в поддержку')
    gr.Markdown('**SHADOW-RESEARCH ONLY:** готовый проект НЕ включает автоотправку: предварительная категория и очередь требуют проверки оператором.')
    gr.Markdown('Демонстрация для английского языка. Введите обезличенное обращение. Решение требует проверки оператором; отправка в систему поддержки не выполняется.')
    gr.Markdown('Предполагаемая категория и решение о ручной проверке показаны отдельно. Ручная проверка не заменяет категорию на OTHER.')
    with gr.Accordion('Что означают категории',open=False):
        gr.Markdown('ACCOUNT — доступ и профиль; CANCELLATION_FEE — стоимость отмены; CONTACT — связь с поддержкой; DELIVERY — способы и общие сроки доставки; FEEDBACK — отзыв/жалоба; INVOICES — счета и платёжные документы; NEWSLETTER — подписка/отписка от рассылки; ORDER — оформление, изменение, отмена и отслеживание конкретного заказа; PAYMENT — способы и ошибки оплаты; REFUNDS — возврат денег; SHIPPING — адрес доставки.')
    session_history = gr.State([])
    with gr.Row():
        with gr.Column():
            user_text = gr.Textbox(lines=4, label='Обращение клиента')
            multiple = gr.Checkbox(label="Распознавать несколько запросов в одном обращении", value=False)
            gr.Examples(examples=[[t] for t in ['I want to cancel my order', "I can't log in", 'Where is my refund?', 'I will sue you', 'What causes the northern lights']], inputs=user_text)
            submit = gr.Button('Анализировать', variant='primary')
            message = gr.Markdown()
        with gr.Column():
            category = gr.Textbox(label='Предполагаемая категория обращения', interactive=False)
            queue = gr.Textbox(label='Рекомендованная очередь', interactive=False)
            status = gr.Textbox(label='Статус', interactive=False)
            priority = gr.Textbox(label='Предварительный приоритет', interactive=False)
            reasons = gr.Textbox(label='Причины решения', interactive=False)
            top3 = gr.Label(label='Top-3 категорий (оценки модели)', num_top_classes=3)
    history_table = gr.Dataframe(headers=HISTORY_COLUMNS, label='История этой сессии (до 100 записей)', interactive=False)
    submit.click(process_web_request, inputs=[user_text,session_history,multiple],
        outputs=[category,queue,status,priority,reasons,top3,history_table,session_history,message])
smoke_ui = process_web_request('Contact me at demo@example.com about my invoice', [])
assert len(smoke_ui[7]) == 1 and 'demo@example.com' not in smoke_ui[7][0]['Текст (маскир.)']
assert process_web_request('', [])[7] == []
# Регрессия найденного дефекта: ручной разбор не должен скрывать предложенную тему.
for case in ['Please email me an invoice for my purchase','Please remove me from your promotional mailing list']:
    direct = sc.route_ticket(case,model)
    rendered = process_web_request(case,[])
    assert rendered[0] == (direct['category'] or 'Не определена')
    assert rendered[0] != 'OTHER'
stage8_9_checks = {'status':'BUILT_AND_HANDLER_CHECKED', 'gradio_version':gr.__version__,
    'per_session_history':True, 'masked_text_smoke':True, 'external_ticket_delivery':False,'operational_auto_enabled':False,
    'public_launch_requested':bool(SHARE_DEMO and LAUNCH_UI), 'real_colab_browser_test':False}
if not LAUNCH_UI:
    # Проверяем настоящий HTML и конфигурацию через ASGI без сетевого сервера.
    from fastapi import FastAPI
    from starlette.testclient import TestClient
    check_app = gr.mount_gradio_app(FastAPI(), demo, path='/', theme=gr.themes.Soft())
    with TestClient(check_app) as client:
        responses = {path:client.get(path) for path in ['/config','/']}
        assert all(response.status_code == 200 for response in responses.values())
        assert 'text/html' in responses['/'].headers['content-type']
    stage8_9_checks['asgi_html_and_config'] = {path:response.status_code for path,response in responses.items()}
    stage8_9_checks['browser_visual_check'] = False
if (REPORTS_DIR/'results.json').exists():
    saved_results = json.loads((REPORTS_DIR/'results.json').read_text(encoding='utf-8'))
    saved_results['stage8_9'] = stage8_9_checks
    sc.write_json(REPORTS_DIR/'results.json', saved_results)
print(stage8_9_checks)
if LAUNCH_UI:
    demo.launch(share=SHARE_DEMO, prevent_thread_lock=True, inline=True, theme=gr.themes.Soft())
else:
    print('Интерфейс создан. Для открытия поменяйте LAUNCH_UI на True и выполните эту ячейку.')


# Выводы по этапам 8-9 (Заключение)

Весь Colab-ноутбук использует одну функцию `sc.route_ticket` в офлайн-оценке, регрессионных тестах и Gradio. Модель показывает предполагаемую категорию и Top-3, но пока **никого автоматически не перенаправляет**. `accepted_category=None` и `decision_category=OTHER` означают отказ от автоматического решения, а не ошибку тематической классификации.

Три обязательных замечания ревью: (1) test Bitext насыщен и плохо различает модели; (2) в прежней версии синтетический OOD-holdout давал **2/24** автоматических решений на нерелевантных обращениях; (3) прежний выбор модели по авторскому DEV-10 **не воспроизводился** на остальных 15 LOIO-интентах. Исправленный код пересчитывает метрики заново и отличает их от исторических результатов.

Проверки угроз, подозрительных платежей, технических проблем, нескольких запросов и недостаточного контекста — демонстрационные правила, **не обученный и не валидированный P1-классификатор**. Флаги P1 должны быть проверены на независимой экспертной разметке с упором на recall. Нужны реальные 300–500 ID, ≥200 OOD и подтверждённые правила очередей. Отложенные `real_blind_id.csv`, `real_blind_ood.csv` и `real_blind_risk.csv` могут быть оценены после фиксации модели, но без проверки происхождения не становятся независимыми по факту загрузки.

Веб-интерфейс не отправляет обращения в CRM, история сессии лишь частично маскирует персональные данные. `joblib` загружает pickle: используйте только артефакты, которым доверяете; SHA256 рядом с файлом не доказывает подлинность. Результат: MVP и исследование в теневом режиме, а не готовая к работе автоматическая поддержка.

# Этапы 10-12: интеграция, тестирование, документация

**Как устроен ноутбук.** Код проекта лежит в пакете `support_router` (папка `src/` репозитория), а ноутбук его импортирует: ячейка 1 больше не записывает модули строками. Функции перенесены из `support_core.py` версии 3.1.1 без изменений, поэтому все вызовы `sc.…` в ячейках этапов 1-9 остались прежними. В этих ячейках изменены только ячейка 1 (откуда брать код), строка импорта в проверке ячейки 45 и явная кодировка UTF-8 в ячейках 1, 45, 46 и в ячейке веб-интерфейса: без неё ноутбук падал на Windows. Модель от переноса не изменилась: скрипт обучения на пакете дал вероятности, совпадающие с моделью ноутбука до последнего знака на 3 442 фразах.

**Как пересчитать.** «Выполнить всё» пересчитывает ноутбук целиком, а ячейка 57 в конце сверяет ключевые цифры отчёта и презентации с новым прогоном.

# Этап 10. Интеграция

Модель, маршрутизация и интерфейс собраны в приложения поверх пакета `support_router`. Они ничего не обучают: загружают модель, сохранённую в ячейке 45, и принимают решения той же функцией `route_ticket`, что и весь ноутбук.

- **HTTP-сервис на FastAPI** (`support_router.api`): `POST /v1/route` и `/v1/route/batch` для CRM, `/health`, `/ready`, паспорт модели `/v1/model` и метрики Prometheus `/metrics`.
- **Интерфейс Gradio** (`app.py`): тот же экран, что на этапе 9, плюс строка теневого режима, раздел «О модели» и слова, которые сильнее всего повлияли на тему. Умеет размечать CSV-файлы.
- **Docker**: `Dockerfile` собирает образ сервиса вместе с моделью, `docker-compose.yml` поднимает его вместе с Prometheus и правилами алертов.

Ниже приложения проверяются в отдельных процессах Python, как будто их запускает другой человек.

In [ ]:
# @title Ячейка 47: Из чего состоит проект
# @markdown Этап 10. Код в пакете `support_router`, рядом два приложения и файлы для Docker и мониторинга.
import ast
modules = []
for path in sorted((PROJECT_DIR/'src/support_router').glob('*.py')):
    tree = ast.parse(path.read_text(encoding='utf-8'))
    doc = (ast.get_docstring(tree) or '').split('\n')[0]
    modules.append({'модуль': path.stem, 'строк': len(path.read_text(encoding='utf-8').splitlines()), 'что делает': doc})
display(pd.DataFrame(modules))
files = ['app.py', 'Dockerfile', 'docker-compose.yml', 'requirements.txt', 'requirements-api.txt', 'pyproject.toml',
         'monitoring/prometheus.yml', 'monitoring/alert_rules.yml', 'monitoring/alert_rules_test.yml',
         'docs/architecture.md', 'docs/api.md', 'docs/runbook.md', 'README.md']
display(pd.DataFrame([{'файл': f, 'есть': (PROJECT_DIR/f).exists()} for f in files]))
assert all((PROJECT_DIR/f).exists() for f in files), 'в проекте не хватает файлов'
from support_router import api as router_api
print('Адреса HTTP-сервиса:', sorted(r.path for r in router_api.create_app(MODELS_DIR).routes
                                    if r.path.startswith(('/v1', '/health', '/ready', '/metrics'))))
print('requirements-api.txt (образ Docker):')
print((PROJECT_DIR/'requirements-api.txt').read_text(encoding='utf-8'))


In [ ]:
# @title Ячейка 48: Проверка приложений в отдельных процессах
# @markdown Проверки снаружи, как будто приложения запускает другой человек: самопроверка Gradio-приложения,
# @markdown разметка CSV-файла, работающий сервер Gradio и HTTP-сервис FastAPI с неверными запросами и метриками.
import socket
from gradio_client import Client
APP_ENV = dict(os.environ, PYTHONPATH=os.pathsep.join([str(PROJECT_DIR/'src'), str(PROJECT_DIR)]),
               GRADIO_ANALYTICS_ENABLED='False',
               NO_PROXY='127.0.0.1,localhost', no_proxy='127.0.0.1,localhost')

# Журналы серверов пишем в файлы, а не в канал: на Windows буфер канала около 4 КБ, и непрочитанный канал
# останавливает сервер посреди проверки.
LOG_DIR = REPORTS_DIR/'server_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def log_tail(path):
    return path.read_text(encoding='utf-8', errors='replace')[-3000:] if path.exists() else ''

def run_app(*args, timeout=600):
    return subprocess.run([sys.executable, 'app.py', *args], cwd=PROJECT_DIR, env=APP_ENV,
                          capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=timeout)

# 1. Самопроверка: модель загружается, хеши сходятся, рабочий режим сам ничего не отправляет.
self_check = run_app('--check')
print(self_check.stdout)
assert self_check.returncode == 0 and 'CHECK OK' in self_check.stdout, self_check.stderr[-3000:]

# 2. Разметка CSV: решения совпадают с теми, что ноутбук получает напрямую, тексты маскируются.
batch_texts = [t for t, _ in scenario_data.DEMO] + [
    "I'm going to kill you", 'Contact me at demo@example.com or +44 20 7946 0958 about my invoice', '', '   ']
batch_in, batch_out = REPORTS_DIR/'app_batch_input.csv', REPORTS_DIR/'app_batch_output.csv'
pd.DataFrame({'id': [f'case-{i:02d}' for i in range(len(batch_texts))], 'text': batch_texts}).to_csv(batch_in, index=False)
batch_run = run_app('--batch', str(batch_in), '--out', str(batch_out))
assert batch_run.returncode == 0, batch_run.stderr[-3000:]
exported = pd.read_csv(batch_out, keep_default_na=False, dtype=str)
live = sc.batch_route(model, batch_texts)
live_shadow = sc.batch_route(sc.shadow_bundle(model), batch_texts)
assert len(exported) == len(batch_texts)
for i, (r, sr) in enumerate(zip(live, live_shadow)):
    row = exported.iloc[i]
    assert row['id'] == f'case-{i:02d}'
    for field in ['category', 'suggested_category', 'queue', 'status', 'priority']:
        assert row[field] == ('' if r[field] is None else r[field]), (i, field)
    assert row['reason_codes'] == ';'.join(r['reason_codes']) and row['shadow_status'] == sr['status']
    if r['confidence'] is not None:
        assert abs(float(row['confidence']) - r['confidence']) < 1e-12
assert not exported.text_masked.str.contains('demo@example.com|7946', regex=True).any()
display(exported[['id', 'text_masked', 'suggested_category', 'status', 'priority', 'shadow_status', 'reason_codes']])

# 3. Сервер: внешняя система шлёт текст и получает тот же ответ, что даёт route_ticket.
def free_port():
    with socket.socket() as probe:
        probe.bind(('127.0.0.1', 0))
        return probe.getsockname()[1]

port = free_port()
gradio_log = LOG_DIR/'gradio.log'
with open(gradio_log, 'wb') as gradio_err:
    server = subprocess.Popen([sys.executable, 'app.py', '--port', str(port)], cwd=PROJECT_DIR, env=APP_ENV,
                              stdout=subprocess.DEVNULL, stderr=gradio_err)
api_texts = ['I want to cancel my order', "I'm going to sue you", 'Где мой заказ?', '',
             'Reset my password and check my refund']
api_rows = []
try:
    client, started = None, time.time()
    while client is None and time.time() - started < 180:
        if server.poll() is not None:
            raise RuntimeError(log_tail(gradio_log))
        try:
            client = Client(f'http://127.0.0.1:{port}/', verbose=False)
        except Exception:
            time.sleep(2)
    assert client is not None, 'сервер не ответил за 3 минуты'
    for text in api_texts:
        remote = client.predict(text, api_name='/route_json')
        local = sc.route_ticket(text, model)
        for field in ['category', 'suggested_category', 'queue', 'status', 'priority', 'reason_codes',
                      'risk_flags', 'model_id', 'policy_version']:
            assert remote[field] == local[field], (text, field)
        api_rows.append({'text': text, 'category': remote['category'], 'status': remote['status'],
                         'priority': remote['priority'], 'risk_flags': remote['risk_flags'],
                         'shadow_status': remote['shadow']['status']})
    shown = client.predict('I want to cancel my order', api_name='/analyze')
    assert shown[0] == (sc.route_ticket('I want to cancel my order', model)['category'] or 'Не определена')
finally:
    server.terminate()
    server.wait(timeout=30)
display(pd.DataFrame(api_rows))


# 4. HTTP-сервис FastAPI: тот же ответ, что route_ticket; неверные запросы получают 422; метрики считаются.
import urllib.error, urllib.request
api_port = free_port()
api_log = LOG_DIR/'fastapi.log'
with open(api_log, 'wb') as api_err:
    api_server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'support_router.api:app', '--host', '127.0.0.1',
                                   '--port', str(api_port)], cwd=PROJECT_DIR, env=APP_ENV,
                                  stdout=subprocess.DEVNULL, stderr=api_err)

def call(method, path, body=None):
    data = None if body is None else json.dumps(body).encode('utf-8')
    req = urllib.request.Request(f'http://127.0.0.1:{api_port}{path}', data=data, method=method,
                                 headers={'content-type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=10) as response:
            return response.status, response.read().decode('utf-8')
    except urllib.error.HTTPError as error:
        return error.code, error.read().decode('utf-8')

fastapi_rows, invalid = [], {}
try:
    started = time.time()
    while True:
        try:
            if call('GET', '/health')[0] == 200:
                break
        except OSError:
            pass
        if api_server.poll() is not None:
            raise RuntimeError(log_tail(api_log))
        if time.time() - started > 120:
            raise RuntimeError('сервис FastAPI не поднялся за 2 минуты')
        time.sleep(1)
    status, body = call('GET', '/ready')
    assert status == 200, body
    for text in api_texts:
        status, body = call('POST', '/v1/route', {'text': text, 'ticket_id': 'notebook-check'})
        remote, local = json.loads(body), sc.route_ticket(text, model)
        assert status == 200
        for field in ['category', 'suggested_category', 'queue', 'status', 'priority', 'reason_codes',
                      'risk_flags', 'model_id', 'policy_version']:
            assert remote[field] == local[field], (text, field)
        fastapi_rows.append({'text': text, 'status': remote['status'], 'priority': remote['priority'],
                             'category': remote['category'], 'shadow_status': remote['shadow']['status']})
    for name, body in [('число вместо текста', {'text': 1}), ('нет поля text', {'txt': 'refund'}),
                       ('лишнее поле', {'text': 'refund', 'vip': True}), ('текст длиннее 5000', {'text': 'a' * 5001})]:
        invalid[name] = call('POST', '/v1/route', body)[0]
    assert set(invalid.values()) == {422}, invalid
    metrics_text = call('GET', '/metrics')[1]
    assert 'router_decisions_total' in metrics_text and 'router_model_loaded 1.0' in metrics_text
finally:
    api_server.terminate()
    api_server.wait(timeout=30)
display(pd.DataFrame(fastapi_rows))
print('FastAPI, неверные запросы:', invalid)

def req_lines(name):
    lines = (PROJECT_DIR/name).read_text(encoding='utf-8').splitlines()
    return [line.strip() for line in lines if line.strip() and not line.lstrip().startswith('#')]

stage10 = {'status': 'APP_BUILT_AND_CHECKED_IN_SEPARATE_PROCESS',
           'app_sha256': sc.sha256_file(PROJECT_DIR/'app.py'),
           'requirements': req_lines('requirements.txt'), 'requirements_api': req_lines('requirements-api.txt'),
           'self_check': True,
           'batch_parity': {'n': len(batch_texts), 'contact_data_masked': True},
           'server_api_parity': {'n': len(api_texts), 'endpoint': '/route_json', 'ui_endpoint': '/analyze'},
           'fastapi_parity': {'n': len(fastapi_rows), 'endpoint': '/v1/route'},
           'fastapi_invalid_requests': invalid,
           'model_id': model.model_id, 'operational_auto_enabled': bool(model.policy.get('auto_enabled')),
           'external_ticket_delivery': False}
saved_results = json.loads((REPORTS_DIR/'results.json').read_text(encoding='utf-8'))
saved_results['stage10'] = stage10
sc.write_json(REPORTS_DIR/'results.json', saved_results)
print(f'Этап 10 готов: самопроверка, {len(batch_texts)} строк CSV, {len(api_texts)} запросов к серверу Gradio и '
      f'{len(fastapi_rows)} к сервису FastAPI дали те же решения, что route_ticket в ноутбуке; неверные запросы '
      f'получили 422. Модель {model.model_id}, автоотправка выключена.')


### Как запустить

- **Сервис в Docker** (на своём компьютере, в папке проекта): `docker compose up --build`, затем http://localhost:8000/docs, Prometheus на http://localhost:9090.
- **Сервис без Docker:** `uvicorn support_router.api:app --port 8000` из папки проекта (нужны `pip install -r requirements.txt` и `pip install -e .`).
- **Интерфейс в Colab:** в новой ячейке `!python app.py --share`. Gradio напечатает временную публичную ссылку; ячейка будет занята, пока сервер работает, остановить его можно кнопкой ■.
- **Разметить файл:** `python app.py --batch tickets.csv --out routed.csv`, в файле нужна колонка `text`.
- **Обучить модель без ноутбука:** `python -m support_router.train`.

# Этап 11. Тестирование

Задача этапа: проверить систему на новых примерах и нестандартных формулировках и исправить найденные ошибки. Проверка идёт в четыре шага.

1. **Автотесты.** Тесты этапов 1-9 и новые: приложение Gradio, нестандартные входы, найденный дефект правил риска, HTTP API (неверные запросы, пустые обращения, маршрутизация, готовность) и метрики с правилами алертов.
2. **Новые фразы.** Два набора написаны 13 ноября для прежней версии проекта, ещё до версии 3.1: 220 обращений по 11 темам (половина намеренно без слов-подсказок вроде refund или invoice) и 100 обращений не по адресу. Наборы заморожены: файлы сверяются по SHA256, по ним ничего не подбирается, и оцениваются они один раз.
3. **Ручная проверка** 30 нестандартных формулировок: опечатки, капс, другой язык, несколько тем, угрозы.
4. **Таблица находок:** что исправлено, что нет и почему.

Новые фразы написаны отдельно от датасета, но не настоящими клиентами. Это честная проверка на незнакомых формулировках, однако не замена реальным данным: до планки ячейки 46 (300 реальных обращений по темам и 200 не по адресу) они не дотягивают ни по объёму, ни по происхождению.

In [ ]:
# @title Ячейка 49: Автотесты
# @markdown Этап 11. Все тесты проекта: этапы 1-9, приложение Gradio, нестандартные входы, найденный дефект
# @markdown правил риска, HTTP API, метрики и правила алертов.
import xml.etree.ElementTree as ET
test_env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD='1', NO_PROXY='127.0.0.1,localhost', no_proxy='127.0.0.1,localhost')
stage11_run = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--color=no', '--disable-warnings',
                              '-p', 'no:cacheprovider', '--junitxml=reports/tests_stage11.xml'],
                             cwd=PROJECT_DIR, env=test_env, capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=1800)
print(stage11_run.stdout[-1200:])
if stage11_run.returncode:
    print(stage11_run.stderr[-3000:])
    raise RuntimeError('Тесты не прошли: этап 11 не засчитываем')
cases = list(ET.parse(REPORTS_DIR/'tests_stage11.xml').getroot().iter('testcase'))
def outcome(case):
    skipped = case.find('skipped')
    if skipped is None:
        return 'прошёл'
    return 'известный пропуск' if 'xfail' in skipped.attrib.get('type', '') else 'пропущен'
by_file = pd.DataFrame([{'файл': c.attrib.get('classname', '').split('.')[-1] + '.py', 'итог': outcome(c)}
                        for c in cases])
by_file = pd.crosstab(by_file['файл'], by_file['итог'], margins=True, margins_name='всего')
display(by_file)
counts = by_file.loc['всего']
stage11_tests = {'tests': len(cases), 'passed': int(counts.get('прошёл', 0)),
                 'skipped': int(counts.get('пропущен', 0)), 'xfailed_known_gaps': int(counts.get('известный пропуск', 0)),
                 'failures': 0, 'errors': 0,
                 'per_file': by_file.drop(index='всего').reset_index().to_dict('records')}


In [ ]:
# @title Ячейка 50: Замороженные наборы новых фраз
# @markdown 220 обращений по 11 темам (половина намеренно без слов-подсказок) и 100 обращений не по адресу.
# @markdown Написаны 13 ноября, до появления версии 3.1, и с тех пор не менялись: SHA256 сверяется.
# @markdown По ним ничего не подбирается и не исправляется; ячейка 51 оценивает их один раз.
FROZEN_SETS = {'hard_test.csv': 'f018ae35067d53b3bfbea7d402389b399cf2ba4956a377e675376aad6743e547',
               'ood_eval.csv': '5e3c4a8327511816c0b3f9a52702796ab3a214efd31533a8267463576eebbd39'}
EVAL_DIR = PROJECT_DIR/'data/eval'
for name, digest in FROZEN_SETS.items():
    assert sc.sha256_file(EVAL_DIR/name) == digest, f'{name} отличается от замороженной версии'
hard = pd.read_csv(EVAL_DIR/'hard_test.csv', keep_default_na=False)
ood_new = pd.read_csv(EVAL_DIR/'ood_eval.csv', keep_default_na=False)
assert len(hard) == 220 and set(hard.category) == set(sc.CATEGORIES) and len(ood_new) == 100

# Пересечения с тем, что модель или её авторы уже видели.
seen_by_team = {}
for name, groups in [('ручной DEV', manual_cases.DEV), ('ручной HOLDOUT', manual_cases.HOLDOUT),
                     ('OOD-dev', scenario_data.DEV), ('OOD-holdout', scenario_data.HOLDOUT)]:
    for texts in groups.values():
        seen_by_team.update({sc.preprocess(t): name for t in texts})
seen_by_team.update({sc.preprocess(t): 'legacy OOD' for t in scenario_data.LEGACY_OOD})
seen_by_team.update({sc.preprocess(t): 'демо' for t, _ in scenario_data.DEMO})
seen_by_team.update({sc.preprocess(x['text']): 'дополнение обучения' for x in category_examples.EXAMPLES})
bitext_texts = set(d.text)
overlap_rows = []
for set_name, frame in [('hard_test', hard), ('ood_eval', ood_new)]:
    for text in frame.text:
        key = sc.preprocess(text)
        if key in bitext_texts or key in seen_by_team:
            overlap_rows.append({'набор': set_name, 'текст': text,
                                 'где уже есть': 'Bitext' if key in bitext_texts else seen_by_team[key]})
print('Совпадений с Bitext:', sum(r['где уже есть'] == 'Bitext' for r in overlap_rows),
      '| с фразами этапов 1-9:', sum(r['где уже есть'] != 'Bitext' for r in overlap_rows))
display(pd.DataFrame(overlap_rows))
if overlap_rows:
    print('Совпадения случайные: это обычные короткие фразы. Каждая меняет итог не больше чем на одну фразу.')
display(hard.groupby(['category', 'difficulty']).size().unstack())
display(ood_new.topic.value_counts().rename('фраз'))


In [ ]:
# @title Ячейка 51: Новые фразы, единственная оценка
# @markdown Сравниваем выбранную модель с базовой (TF-IDF по словам + логистическая регрессия) на одних и тех же фразах.
# @markdown Рабочий режим проверяем на главное обещание: сам ничего не отправляет. Теневой режим показывает,
# @markdown что было бы при включённой автоотправке с порогами этапа 7 (уверенность 0.85, сходство 0.45).
shadow_live = sc.shadow_bundle(model)
DIFFICULTY_RU = {'natural': 'обычные формулировки', 'no_keywords': 'без слов-подсказок'}
TOPIC_RU = {'off_topic': 'вопросы не о магазине', 'out_of_scope': 'о магазине, но вне 11 тем',
            'app_failure': 'сбой сайта или приложения', 'gibberish': 'бессмыслица',
            'other_language': 'другой язык', 'multiple_topics': 'несколько тем сразу'}
EXPECTED_RULE = {'other_language': 'POSSIBLY_UNSUPPORTED_LANGUAGE',
                 'app_failure': 'TECHNICAL_ISSUE_NEEDS_REVIEW',
                 'multiple_topics': 'POSSIBLE_MULTIPLE_REQUESTS'}
ALLOWED_DEPLOYED = ['MANUAL_REVIEW', 'NEEDS_CLARIFICATION']

# --- 220 обращений по 11 темам ---
hard_texts = hard.text.tolist()
hard_pred = np.asarray(model.predict(hard_texts))
hard_base = np.asarray(baseline_final.predict(hard_texts))
hard_routes = sc.batch_route(model, hard_texts)
hard_shadow = sc.batch_route(shadow_live, hard_texts)
hard_report = hard.assign(
    prediction=hard_pred, baseline_prediction=hard_base,
    correct=hard_pred == hard.category.to_numpy(), baseline_correct=hard_base == hard.category.to_numpy(),
    confidence=[r['confidence'] for r in hard_shadow], similarity=[r['similarity'] for r in hard_shadow],
    deployed_status=[r['status'] for r in hard_routes], shadow_status=[r['status'] for r in hard_shadow],
    shadow_queue=[r['queue'] for r in hard_shadow], reasons=[';'.join(r['reason_codes']) for r in hard_shadow])
hard_report['true_queue'] = hard_report.category.map(sc.CATEGORY_TO_QUEUE)
hard_report['shadow_auto'] = hard_report.shadow_status.eq('AUTO_RECOMMENDED')
hard_report['shadow_wrong_queue'] = hard_report.shadow_auto & hard_report.shadow_queue.ne(hard_report.true_queue)
assert hard_report.deployed_status.isin(ALLOWED_DEPLOYED).all(), 'рабочий режим не должен отправлять сам'
hard_report.to_csv(REPORTS_DIR/'stage11_hard_test_predictions.csv', index=False)


def hard_summary(part):
    auto = part.shadow_auto
    wrong = int(part.shadow_wrong_queue.sum())
    return {'n': len(part), 'model_correct': int(part.correct.sum()),
            'model_accuracy': float(part.correct.mean()),
            'baseline_correct': int(part.baseline_correct.sum()),
            'baseline_accuracy': float(part.baseline_correct.mean()),
            'deployed_auto': int(part.deployed_status.eq('AUTO_RECOMMENDED').sum()),
            'shadow_auto': int(auto.sum()), 'shadow_auto_right_queue': int(auto.sum()) - wrong,
            'shadow_wrong_queue': wrong, 'shadow_wrong_category': int((auto & ~part.correct).sum()),
            'shadow_manual': int((~auto).sum()),
            'shadow_wrong_queue_wilson_upper': sc.wilson_upper(wrong, int(auto.sum()))}


hard_results = {'all': hard_summary(hard_report)}
for level, part in hard_report.groupby('difficulty'):
    hard_results[level] = hard_summary(part)
hard_results['paired_vs_baseline'] = sc.paired_error_test(hard.category.to_numpy(), hard_base, hard_pred)
hard_results['per_category'] = {
    cat: {level: {'model': int(g.correct.sum()), 'baseline': int(g.baseline_correct.sum()), 'n': len(g)}
          for level, g in part.groupby('difficulty')}
    for cat, part in hard_report.groupby('category')}

hard_table = pd.DataFrame({DIFFICULTY_RU.get(k, 'все 220'): {
    'фраз': v['n'],
    'тема верна: модель': f"{v['model_correct']} ({v['model_accuracy']:.1%})",
    'тема верна: базовая': f"{v['baseline_correct']} ({v['baseline_accuracy']:.1%})",
    'рабочий режим: сам': v['deployed_auto'],
    'теневой: сам, верная очередь': v['shadow_auto_right_queue'],
    'теневой: сам, чужая очередь': v['shadow_wrong_queue'],
    'теневой: к оператору': v['shadow_manual']} for k, v in hard_results.items()
    if k in ['natural', 'no_keywords', 'all']}).T
display(hard_table)
print('Модель против базовой на тех же фразах (точный биномиальный тест по расхождениям):',
      hard_results['paired_vs_baseline'])
per_category_table = pd.DataFrame({
    cat: {f'{DIFFICULTY_RU[level]}: модель': f"{x['model']}/{x['n']}" for level, x in levels.items()} |
         {f'{DIFFICULTY_RU[level]}: базовая': f"{x['baseline']}/{x['n']}" for level, x in levels.items()}
    for cat, levels in hard_results['per_category'].items()}).T
display(per_category_table)
wrong_auto = hard_report[hard_report.shadow_wrong_queue]
print('Теневой режим отправил бы сам в чужую очередь:')
display(wrong_auto[['text', 'category', 'prediction', 'difficulty', 'confidence', 'similarity']])

# --- 100 обращений не по адресу ---
ood_texts_new = ood_new.text.tolist()
ood_routes_new = sc.batch_route(model, ood_texts_new)
ood_shadow_new = sc.batch_route(shadow_live, ood_texts_new)
ood_report_new = ood_new.assign(
    suggested_category=[r['suggested_category'] for r in ood_shadow_new],
    confidence=[r['confidence'] for r in ood_shadow_new], similarity=[r['similarity'] for r in ood_shadow_new],
    deployed_status=[r['status'] for r in ood_routes_new], shadow_status=[r['status'] for r in ood_shadow_new],
    shadow_queue=[r['queue'] for r in ood_shadow_new], reasons=[';'.join(r['reason_codes']) for r in ood_shadow_new])
ood_report_new['false_auto'] = ood_report_new.shadow_status.eq('AUTO_RECOMMENDED')
ood_report_new['rule_fired'] = [(EXPECTED_RULE[t] in r['reason_codes']) if t in EXPECTED_RULE else None
                                for t, r in zip(ood_new.topic, ood_shadow_new)]
assert ood_report_new.deployed_status.isin(ALLOWED_DEPLOYED).all(), 'рабочий режим не должен отправлять сам'
ood_report_new.to_csv(REPORTS_DIR/'stage11_ood_predictions.csv', index=False)
ood_results = {'n': len(ood_report_new), 'deployed_auto': 0,
               'shadow_false_auto': int(ood_report_new.false_auto.sum()),
               'shadow_false_auto_wilson_upper': sc.wilson_upper(int(ood_report_new.false_auto.sum()),
                                                                 len(ood_report_new)),
               'by_topic': {}}
for topic, part in ood_report_new.groupby('topic'):
    ood_results['by_topic'][topic] = {
        'n': len(part), 'shadow_false_auto': int(part.false_auto.sum()),
        'rule': EXPECTED_RULE.get(topic),
        'rule_fired': int(part.rule_fired.sum()) if topic in EXPECTED_RULE else None}
ood_table = pd.DataFrame({TOPIC_RU[t]: {
    'фраз': v['n'], 'теневой режим отправил бы сам': v['shadow_false_auto'],
    'нужное правило сработало': '' if v['rule'] is None else f"{v['rule_fired']} из {v['n']}"}
    for t, v in ood_results['by_topic'].items()}).T
display(ood_table)
print('Не по адресу, но теневой режим отправил бы сам:')
display(ood_report_new[ood_report_new.false_auto][['text', 'topic', 'suggested_category', 'confidence', 'similarity']])

# --- Графики ---
FIG_DIR = REPORTS_DIR/'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
INK, INK_2, MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#fcfcfb'
BLUE, ORANGE, GRAY = '#2a78d6', '#eb6834', '#c3c2b7'
style = {'axes.spines.top': False, 'axes.spines.right': False, 'axes.spines.left': False,
         'axes.edgecolor': GRAY, 'axes.facecolor': SURFACE, 'figure.facecolor': SURFACE,
         'savefig.facecolor': SURFACE, 'axes.titlelocation': 'left', 'axes.titleweight': 'bold',
         'axes.titlesize': 12, 'axes.titlecolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
         'xtick.labelcolor': INK_2, 'ytick.labelcolor': INK_2, 'text.color': INK, 'font.size': 10}


def finish(ax, title, xlabel):
    ax.xaxis.grid(True, color=GRID, linewidth=1)
    ax.yaxis.grid(False)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    ax.set_xlabel(xlabel, color=INK_2)
    ax.set_title(title, pad=30)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3, frameon=False, fontsize=9,
              handlelength=1.2, borderaxespad=0.2)


with plt.rc_context(style):
    fig, ax = plt.subplots(figsize=(8.2, 3.1))
    rows = ['natural', 'no_keywords']
    ys = np.arange(len(rows))
    for offset, key, color, label in [(-.18, 'baseline_accuracy', GRAY, 'базовая модель'),
                                      (.18, 'model_accuracy', BLUE, 'выбранная модель')]:
        values = [hard_results[r][key] * 100 for r in rows]
        ax.barh(ys + offset, values, height=.32, color=color, label=label, zorder=3)
        for y, v in zip(ys + offset, values):
            ax.text(v + 1, y, f'{v:.0f}%', va='center', color=INK, fontsize=10)
    ax.set_yticks(ys, [f"{DIFFICULTY_RU[r]}\n({hard_results[r]['n']} фраз)" for r in rows])
    ax.invert_yaxis()
    ax.set_xlim(0, 105)
    finish(ax, 'Новые фразы: модель узнаёт слова, а не смысл', 'верно угаданная тема, %')
    fig.tight_layout()
    fig.savefig(FIG_DIR/'stage11_topic_accuracy.png', dpi=160, bbox_inches='tight')
    plt.show()

    fig, ax = plt.subplots(figsize=(8.2, 3.4))
    segments = [('сам, в верную очередь', 'верно', BLUE, 'white'), ('к оператору', 'к оператору', GRAY, INK),
                ('сам, ошибка', 'ошибки', ORANGE, 'white')]
    bars = [('обычные формулировки', [hard_results['natural'][k] for k in
                                      ['shadow_auto_right_queue', 'shadow_manual', 'shadow_wrong_queue']]),
            ('без слов-подсказок', [hard_results['no_keywords'][k] for k in
                                    ['shadow_auto_right_queue', 'shadow_manual', 'shadow_wrong_queue']]),
            ('не по адресу', [0, ood_results['n'] - ood_results['shadow_false_auto'],
                              ood_results['shadow_false_auto']])]
    ys = np.arange(len(bars))
    left = np.zeros(len(bars))
    small = {y: [] for y in ys}
    for idx, (label, short, color, text_color) in enumerate(segments):
        values = np.array([b[1][idx] for b in bars], dtype=float)
        ax.barh(ys, values, left=left, color=color, height=.55, label=label, edgecolor=SURFACE,
                linewidth=2, zorder=3)
        for y, l0, v in zip(ys, left, values):
            if v >= 8:
                ax.text(l0 + v / 2, y, f'{int(v)}', ha='center', va='center', color=text_color, fontsize=10)
            elif v > 0:
                small[y].append(f'{short}: {int(v)}')
        left += values
    for y, total in zip(ys, left):
        if small[y]:
            ax.text(total + 1.5, y, ', '.join(small[y]), va='center', color=INK_2, fontsize=9)
    ax.set_yticks(ys, [f'{b[0]}\n({int(sum(b[1]))})' for b in bars])
    ax.invert_yaxis()
    ax.set_xlim(0, max(left) * 1.3)
    finish(ax, 'Что было бы, если включить автоотправку (теневой режим)', 'обращений')
    fig.tight_layout()
    fig.savefig(FIG_DIR/'stage11_shadow_outcomes.png', dpi=160, bbox_inches='tight')
    plt.show()
print('Рабочий режим на всех 320 фразах: сам не отправил ни одной, всё ушло оператору или попросило текст.')


In [ ]:
# @title Ячейка 52: Ручная проверка нестандартных формулировок
# @markdown Каждая строка: что проверяем, текст, какую тему ждём (или что обращение обязано уйти оператору)
# @markdown и какое правило должно сработать. «Пройдено» означает: рабочий режим сам не отправил, теневой не
# @markdown отправил бы в чужую очередь, нужное правило сработало. Отдельно видно, угадана ли тема.
LONG_STORY = ('Hello, I have been a customer of yours for many years and I have always been happy. '
              'Last month my sister told me about a concert in our city, so I bought two tickets for us '
              'on your website. We were both very excited and even booked a hotel nearby. Yesterday I '
              'received a message saying that the concert was cancelled because the singer is ill. I '
              'understand that these things happen and nobody is to blame. Still, the tickets were '
              'expensive, so I would like to get a refund as soon as possible.')
POLITE = ('Hello, I hope you are doing well. I was wondering if you could possibly help me, I would '
          'like to get an invoice for my purchase. Thank you very much!')
# (группа, что проверяем, текст, ожидаемая тема или None, обязательно к оператору, ожидаемый флаг или код)
NONSTANDARD = [
    ('форма текста', 'капс и восклицания', 'I WANT A REFUND NOW!!!', 'REFUNDS', False, None),
    ('форма текста', 'лишние пробелы и переносы', '  I   want\n to cancel \t my order  ', 'ORDER', False, None),
    ('форма текста', 'опечатки', 'i wnat to chnage my shiping adress', 'SHIPPING', False, None),
    ('форма текста', 'опечатки и капс', 'CANT LOGIN TO MY ACOUNT!!!', 'ACCOUNT', False, None),
    ('форма текста', 'сокращения', 'pls cancel my order asap thx', 'ORDER', False, None),
    ('форма текста', 'смайлики', 'where is my order 😡😡', 'ORDER', False, None),
    ('форма текста', 'вежливо и длинно', POLITE, 'INVOICES', False, None),
    ('форма текста', 'одно слово', 'refund', 'REFUNDS', False, None),
    ('форма текста', 'короткая жалоба', "I can't log in", 'ACCOUNT', False, None),
    ('форма текста', 'шаблон вместо номера', 'I have a problem with order {{Order Number}}', 'ORDER', False, None),
    ('форма текста', 'номер заказа цифрами', 'my order 48213 has not arrived yet', 'ORDER', False, None),
    ('форма текста', 'отрицание', 'I do not want a refund, I want to change my order', 'ORDER', False, None),
    ('форма текста', 'HTML-разметка', '<b>cancel</b> my order please', 'ORDER', False, None),
    ('форма текста', 'длинная история', LONG_STORY, 'REFUNDS', False, None),
    ('форма текста', 'эмоции без просьбы', 'you are the worst company ever', 'FEEDBACK', False, None),
    ('не по адресу', 'приветствие без просьбы', 'hello', None, True, None),
    ('не по адресу', 'вопрос не о магазине', 'What is the capital of France?', None, True, None),
    ('не по адресу', 'русский язык', 'Где мой заказ?', None, True, 'POSSIBLY_UNSUPPORTED_LANGUAGE'),
    ('не по адресу', 'смесь языков', 'Привет, I need a refund', None, True, 'POSSIBLY_UNSUPPORTED_LANGUAGE'),
    ('не по адресу', 'испанский язык', 'Quiero cancelar mi pedido', None, True, 'POSSIBLY_UNSUPPORTED_LANGUAGE'),
    ('не по адресу', 'только знаки', '?!?!', None, True, 'INSUFFICIENT_SUPPORTED_TEXT'),
    ('не по адресу', 'похоже на код', "'; DROP TABLE tickets; --", None, True, None),
    ('не по адресу', 'только ссылка', 'https://example.com/help', None, True, None),
    ('несколько тем', 'две темы', 'My payment failed and I need to change my shipping address', None, True,
     'POSSIBLE_MULTIPLE_REQUESTS'),
    ('несколько тем', 'три темы', 'Reset my password, return my money, and cancel my order', None, True,
     'POSSIBLE_MULTIPLE_REQUESTS'),
    ('риски', 'угроза через «I\'m»', "I'm going to kill you", None, True, 'POSSIBLE_THREAT'),
    ('риски', 'иск через «gonna»', "I'm gonna sue you", None, True, 'POSSIBLE_LEGAL_CLAIM'),
    ('риски', 'отрицание угрозы', "I'm not going to hurt anyone, I just want my refund", 'REFUNDS', False,
     'NO_RISK_FLAG'),
    ('риски', 'взлом другим порядком слов', 'Someone hacked my account and ordered things I never asked for',
     None, True, 'POSSIBLE_FRAUD'),
    ('риски', 'мошенничество новыми словами', 'I see a fraudulent charge on my card', None, True, 'POSSIBLE_FRAUD'),
]
manual_rows = []
for group, kind, text, expected, must_operator, signal in NONSTANDARD:
    r, sh = sc.route_ticket(text, model), sc.route_ticket(text, shadow_live)
    shadow_auto = sh['status'] == 'AUTO_RECOMMENDED'
    if must_operator or expected is None:
        shadow_ok = not shadow_auto
    else:
        shadow_ok = not shadow_auto or sh['queue'] == sc.CATEGORY_TO_QUEUE[expected]
    signals = set(r['risk_flags']) | set(r['reason_codes'])
    if signal == 'NO_RISK_FLAG':
        signal_ok = not {'POSSIBLE_THREAT', 'POSSIBLE_FRAUD', 'POSSIBLE_LEGAL_CLAIM'} & set(r['risk_flags'])
    else:
        signal_ok = signal is None or signal in signals
    manual_rows.append({
        'группа': group, 'что проверяем': kind, 'текст': text if len(text) < 70 else text[:67] + '...',
        'ждём': expected or 'к оператору', 'модель думает': r['suggested_category'] or 'нет',
        'уверенность': None if r['confidence'] is None else round(r['confidence'], 3),
        'приоритет': r['priority'], 'рабочий режим': r['status'],
        'теневой режим': f"сам: {sh['queue']}" if shadow_auto else sh['status'],
        'тема': '' if expected is None else ('✓' if r['suggested_category'] == expected else '✗'),
        'правило': '' if signal is None else ('✓' if signal_ok else '✗'),
        'пройдено': r['status'] != 'AUTO_RECOMMENDED' and shadow_ok and signal_ok})
manual_report = pd.DataFrame(manual_rows)
manual_report.to_csv(REPORTS_DIR/'stage11_nonstandard_cases.csv', index=False)
with pd.option_context('display.max_colwidth', 70):
    display(manual_report)
nonstandard_results = {
    'n': len(manual_report), 'passed': int(manual_report['пройдено'].sum()),
    'topic_expected_n': int(manual_report['тема'].ne('').sum()),
    'topic_correct': int(manual_report['тема'].eq('✓').sum()),
    'failed': manual_report.loc[~manual_report['пройдено'], ['что проверяем', 'текст']].to_dict('records')}
print(f"Пройдено {nonstandard_results['passed']} из {nonstandard_results['n']}; "
      f"тема угадана в {nonstandard_results['topic_correct']} из {nonstandard_results['topic_expected_n']} "
      'случаев, где у обращения одна понятная тема.')


In [ ]:
# @title Ячейка 53: Что нашли и что исправили
# @markdown Сравниваем текущие правила рисков с версией 3.1.0.
# @markdown Результаты этапа 11 относятся к стандартному режиму. Отдельная проверка режима распознавания нескольких запросов приведена в reports/review_benchmark.json.
# @markdown Ниже доказательство, что она ничего не меняет на данных проекта, и полная таблица находок.

def risk_rules_v310(text):
    """Правила рисков версии 3.1.0 без изменений: только для сравнения «до и после»."""
    t = sc.preprocess(text)
    quoted = re.sub(r'"[^"\n]*"|“[^”\n]*”', " ", t)
    reported = bool(re.search(r"\b(threatened me|threatening me|sent me (?:a )?threat|someone threatened)\b", t))
    direct = bool(re.search(r"\bi(?: will|'ll| am going to| intend to)\s+"
                            r"(?:kill|hurt|attack|shoot|make)\s+"
                            r"(?:you|your|the courier|the driver|staff|someone|you pay)\b", quoted))
    direct = direct or bool(re.search(r"\bi(?: will|'ll)\s+make you pay\b", quoted))
    fraud = bool(re.search(r"\b(unauthori[sz]ed (?:payment|charge|transaction)|(?:account|card) (?:was |has been )?(?:hacked|stolen|compromised)|did not authori[sz]e|do not recogni[sz]e (?:this|the|a) (?:charge|payment))\b", t))
    legal = bool(re.search(r"\b(i(?: will|'ll| intend to| am going to) (?:sue|take (?:you|this) to court)|formal (?:legal )?(?:claim|notice)|legal (?:notice|proceedings)|my lawyer (?:has|will|is))\b", quoted))
    abuse = bool(re.search(r"\b(?:idiots?|morons?|stupid|fuck(?:ing)?|shit|assholes?)\b", quoted))
    fraud = fraud or bool(re.search(r"\b(?:i (?:did not|didn't|never) (?:make|place|approve) "
                                  r"(?:this|that|the) (?:purchase|order|payment|transaction)|"
                                  r"someone (?:used|is using) my (?:card|account)|"
                                  r"this (?:purchase|charge|payment) (?:is not|was not) mine)\b", t))
    return [f for f, on in [('POSSIBLE_THREAT', direct or reported), ('POSSIBLE_FRAUD', fraud),
                            ('POSSIBLE_LEGAL_CLAIM', legal), ('ABUSE_LANGUAGE', abuse)] if on]


project_texts = raw.utterance.tolist()
for groups in [manual_cases.DEV, manual_cases.HOLDOUT, scenario_data.DEV, scenario_data.HOLDOUT]:
    for texts in groups.values():
        project_texts += list(texts)
project_texts += list(scenario_data.LEGACY_OOD) + [t for t, _ in scenario_data.DEMO]
project_texts += [x['text'] for x in category_examples.EXAMPLES]
changed_on_project = [t for t in project_texts if risk_rules_v310(t) != sc.detect_signals(t)['risk_flags']]
changed_on_frozen = [t for t in hard.text.tolist() + ood_new.text.tolist()
                     if risk_rules_v310(t) != sc.detect_signals(t)['risk_flags']]
n_project = f'{len(project_texts):,}'.replace(',', ' ')
if not changed_on_project and not changed_on_frozen:
    print(f'Правки 3.1.1–3.1.2 не меняет ни одного флага ни на {n_project} фразах проекта '
          '(весь Bitext и все наборы этапов 1-9), ни на замороженных наборах.')
else:
    print(f'Правки 3.1.1–3.1.2 меняет флаги у {len(changed_on_project)} из {n_project} фраз проекта '
          f'и у {len(changed_on_frozen)} фраз замороженных наборов.')
assert not changed_on_project, changed_on_project[:5]

fixed_examples = ["I'm going to kill you", "I'm gonna sue you", 'The courier threatened to hurt me',
                  'Someone hacked my account', 'My card got hacked', 'I never placed this order',
                  "This transaction isn't mine", "There is a charge I don't recognise on my card"]
fixed_examples += ['I see a fraudulent charge on my card', 'I have been scammed by your seller',
                 "I'm gonna kill ya", 'i m going to kill you']
open_examples = []
display(pd.DataFrame([{'фраза': t, 'было (3.1.0)': ', '.join(risk_rules_v310(t)) or 'нет флага',
                       'стало (3.1.2)': ', '.join(sc.detect_signals(t)['risk_flags']) or 'нет флага',
                       'приоритет': sc.route_ticket(t, model)['priority']}
                      for t in fixed_examples + open_examples]))

hr, orr, bt = hard_results, ood_results, ood_results['by_topic']
nk = hr['no_keywords']
def operator_note(topic):
    info = bt[topic]
    if info['shadow_false_auto']:
        return f"теневой режим отправил бы сам {info['shadow_false_auto']} из {info['n']}"
    return (f"все {info['n']} всё равно ушли бы к оператору: {info['rule_fired']} по правилу, остальные потому, "
            "что модель не уверена или фраза не похожа на обучающие")


tech_note, lang_note = operator_note('app_failure'), operator_note('other_language')
findings = [
    {'что нашли': 'Угрозы, иски и мошенничество в другом написании не получали флаг риска: '
                  '«I\'m going to kill you», «gonna sue», «threatened to hurt me», «someone hacked my account», '
                  '«I never placed this order», «a charge I don\'t recognise»',
     'где': 'ручная проверка', 'статус': 'исправлено',
     'что сделали': 'Версия 3.1.1, модуль rules: в существующие шаблоны добавлены сокращения, другой порядок слов, '
                    'прошедшее время и синоним на том же месте. На фразах проекта флаги не изменились, '
                    'выводы этапов 1-9 тоже. Добавлены регрессионные тесты.'},
    {'что нашли': 'Ложный P1 на «I will make you a list» и пропуски прямых сообщений о мошенничестве и угрозах.',
     'где': 'ручная проверка', 'статус': 'исправлено',
     'что сделали': 'Сузили ветку make you pay, добавили варианты прямых сообщений и парные отрицательные примеры. '
                    'Синтетический набор data/eval/risk_regression.csv проверяет регрессии по каждому флагу.'},
    {'что нашли': f"Без слов-подсказок модель угадывает тему в {nk['model_correct']} из {nk['n']} фраз, "
                  + ('базовая столько же' if nk['baseline_correct'] == nk['model_correct']
                     else f"базовая в {nk['baseline_correct']}"),
     'где': 'замороженный набор 220', 'статус': 'ограничение модели',
     'что сделали': f"TF-IDF узнаёт слова, а не смысл. Защищает порог: теневой режим отправил бы сам только "
                    f"{nk['shadow_auto']} из {nk['n']} таких фраз, остальные ушли бы оператору."},
    {'что нашли': f"Две темы в одном обращении: правило сработало на {bt['multiple_topics']['rule_fired']} из 15, "
                  f"теневой режим отправил бы сам {bt['multiple_topics']['shadow_false_auto']}",
     'где': 'набор «не по адресу»', 'статус': 'открыто',
     'что сделали': 'Не исправлено: расширение словаря тем меняет отбор обращений для порога на этапе 7. '
                    'В рабочем режиме такие обращения всё равно видит оператор.'},
    {'что нашли': f"Сбои сайта и приложения: правило сработало на {bt['app_failure']['rule_fired']} из 15",
     'где': 'набор «не по адресу»', 'статус': 'открыто',
     'что сделали': f'Не исправлено: слова вроде error встречаются и в обычных вопросах об оплате. '
                    f'Из этих обращений {tech_note}.'},
    {'что нашли': f"Другой язык распознан в {bt['other_language']['rule_fired']} из 15: правило знает только "
                  'несколько алфавитов и десяток слов',
     'где': 'набор «не по адресу»', 'статус': 'открыто',
     'что сделали': f'Не исправлено, для сервиса нужен настоящий определитель языка. Из этих обращений {lang_note}.'},
    {'что нашли': f"Теневой режим отправил бы сам {orr['shadow_false_auto']} из {orr['n']} обращений не по адресу "
                  f"и {hr['all']['shadow_wrong_queue']} из {hr['all']['n']} обычных в чужую очередь",
     'где': 'оба замороженных набора', 'статус': 'учтено',
     'что сделали': 'Автоотправка остаётся выключенной, как решили на этапе 7: сначала реальные обращения '
                    '(не меньше 300 по темам и 200 не по адресу) и согласование с бизнесом.'},
    {'что нашли': 'Короткие и неоднозначные фразы: «I can\'t log in» модель относит к INVOICES, '
                  'шаблон {{Order Number}} к PAYMENT, «I do not want a refund…» к REFUNDS',
     'где': 'ручная проверка', 'статус': 'ограничение модели',
     'что сделали': 'Уверенность у таких ответов низкая, поэтому даже теневой режим отдаёт их оператору. '
                    'Лечится данными, а не правилами.'},
]
findings_table = pd.DataFrame(findings)[['что нашли', 'где', 'что сделали', 'статус']]
with pd.option_context('display.max_colwidth', 400):
    display(findings_table)
findings_table.to_csv(REPORTS_DIR/'stage11_findings.csv', index=False)

stage11 = {
    'code_version': sc.VERSION,
    'fix': {'what': 'Risk rule variants, first-person fraud reports and narrower make-you-pay matching',
            'project_texts_checked': len(project_texts), 'project_texts_changed': len(changed_on_project),
            'frozen_texts_changed': len(changed_on_frozen),
            'fixed_examples': fixed_examples, 'open_examples': open_examples},
    'tests': stage11_tests,
    'frozen_sets': {name: {'sha256': digest, 'written': '2026-11-13', 'role': 'post-selection diagnostic'}
                    for name, digest in FROZEN_SETS.items()},
    'frozen_overlap_with_project': overlap_rows,
    'hard_test': hard_results, 'ood_new': ood_results, 'nonstandard_manual': nonstandard_results,
    'findings': findings, 'independence_note': 'Phrases written by the project assistant, not real customers; '
    'below the 300 ID / 200 OOD real-data bar of results.json external_validation.'}
saved_results = json.loads((REPORTS_DIR/'results.json').read_text(encoding='utf-8'))
saved_results['stage11'] = stage11
sc.write_json(REPORTS_DIR/'results.json', saved_results)
print('Итоги этапа 11 записаны в reports/results.json (раздел stage11).')


In [ ]:
# @title Ячейка 54: Выводы этапа 11
# @markdown Текст собирается из результатов этого прогона, поэтому цифры в нём всегда совпадают с таблицами выше.
from IPython.display import Markdown
hr, orr, ns, tests = hard_results, ood_results, nonstandard_results, stage11_tests
nat, nk = hr['natural'], hr['no_keywords']
paired = hr['paired_vs_baseline']
total_new = hr['all']['n'] + orr['n']
skip_note = f", {tests['skipped']} пропущены без семантического энкодера" if tests['skipped'] else ''


def plural(value, one, few, many):
    k = abs(int(value)) % 100
    if 11 <= k <= 14:
        return many
    return {1: one, 2: few, 3: few, 4: few}.get(k % 10, many)


num_ru = lambda x: f'{x:,}'.replace(',', ' ')


def compare(model_n, base_n):
    if model_n == base_n:
        return f'столько же, сколько базовая ({base_n})'
    return f'базовая угадала {base_n}'


shadow_auto_all = hr['all']['shadow_auto']
errors_seen = hr['all']['shadow_wrong_queue'] + orr['shadow_false_auto']
errors_note = ('Ошибок немного, но на нескольких сотнях фраз нельзя подтвердить уровень ошибок ниже 1%, '
               'которого требует этап 7.' if errors_seen else
               'Ошибок нет, но на нескольких сотнях фраз нельзя подтвердить уровень ошибок ниже 1%, '
               'которого требует этап 7.')
test_f1 = json.loads((REPORTS_DIR/'results.json').read_text(encoding='utf-8'))['test']['macro_f1']
summary_md = f"""
### Выводы этапа 11

**Главное обещание выполнено.** В рабочем режиме система не отправила сама ни одного из {total_new} новых
обращений: каждое ушло оператору с подсказкой темы или с просьбой прислать текст. Из {tests['tests']}
автотестов прошли {tests['passed']}, ни один не упал; {tests['xfailed_known_gaps']} записаны как известные пропуски
правил рисков{skip_note}.

**Модель узнаёт слова, а не смысл.** Когда клиент пишет обычными словами, тема угадана в
{nat['model_correct']} из {nat['n']} обращений ({nat['model_accuracy']:.0%}), {compare(nat['model_correct'], nat['baseline_correct'])}.
Когда слов-подсказок нет, модель угадывает только {nk['model_correct']} из {nk['n']}, {compare(nk['model_correct'], nk['baseline_correct'])}.
По всем 220 фразам модель права там, где базовая ошиблась, {paired['a_wrong_b_correct']} {plural(paired['a_wrong_b_correct'], 'раз', 'раза', 'раз')}, а наоборот
{paired['a_correct_b_wrong']} {plural(paired['a_correct_b_wrong'], 'раз', 'раза', 'раз')} (p = {paired['p_value']:.2f}), то есть на новых фразах разница с базовой
моделью статистически не доказана. Цифры test Bitext (macro-F1 {test_f1:.3f}) этого не показывают: test собран
из тех же шаблонов, что и обучение.

**Выключенная автоотправка оправдана.** В теневом режиме, то есть если бы автоотправку включили с порогами
этапа 7, система сама отправила бы {shadow_auto_all} из {hr['all']['n']} обычных обращений, из них
{hr['all']['shadow_wrong_queue']} в чужую очередь, и ещё {orr['shadow_false_auto']} из {orr['n']} обращений
не по адресу. {errors_note}

**Правила приоритета.** Нейтральная фраза «I will make you a list» остаётся P3.
Прямое сообщение «I see a fraudulent charge on my card» получает флаг мошенничества и P1.
Регрессионные примеры проверяют конкретные формулировки. Полнота на реальном потоке неизвестна.

Ручная проверка: пройдено {ns['passed']} из {ns['n']} нестандартных формулировок, тема угадана в
{ns['topic_correct']} из {ns['topic_expected_n']} случаев с одной понятной темой. Не пройденные строки
описаны в таблице находок.
"""
display(Markdown(summary_md))


# Этап 13. Аудит: независимая оценка, стоимость ошибок, калибровка и обратная связь

Этап 11 — исторический прогон версии 3.1.2 на авторских наборах. После аудита добавлены:

- замороженный независимый набор `independent-v1-2026-11-23`;
- cost-sensitive правило по полной матрице потерь `R(a|x)`;
- калибровка по категориям и risk–coverage на независимом наборе;
- отдельные метрики каскада, риск-правил и многотемных обращений;
- контур обратной связи оператора и релизный манифест.

Числа ниже читаются из отчётов, а не вводятся вручную. Если набор изменён после фиксации, проверка SHA-256 остановит ячейку.


In [ ]:
# @title Ячейка 58a: Режим выполнения этапа 13
# @markdown `verified_reports` только читает и проверяет отчёты; `full_pipeline` пересчитывает benchmark (медленно).
RUN_MODE = globals().get('RUN_MODE', 'verified_reports')
assert RUN_MODE in {'verified_reports', 'full_pipeline'}, 'RUN_MODE должен быть verified_reports или full_pipeline'
print('RUN_MODE =', RUN_MODE)


In [ ]:
# @title Ячейка 59: Независимый замороженный набор
from support_router import independent_cases
from support_router.identity import validate_report_identity

RUN_MODE = globals().get('RUN_MODE', 'verified_reports')
assert RUN_MODE in {'verified_reports', 'full_pipeline'}
if RUN_MODE == 'full_pipeline':
    import subprocess, sys
    subprocess.run([sys.executable, str(PROJECT_DIR/'scripts/run_independent_benchmark.py')],
                   check=True, cwd=PROJECT_DIR)

M = json.loads((MODELS_DIR / 'router_meta.json').read_text(encoding='utf-8'))
IB = json.loads((REPORTS_DIR / 'independent_benchmark.json').read_text(encoding='utf-8'))
assert IB['protocol']['sha256'] == independent_cases.sha256(), 'Набор изменён после фиксации'
assert IB['protocol']['sha256'] == IB['protocol']['frozen_sha256']
IDENTITY = validate_report_identity(M, IB)
print('Версия набора:', IB['protocol']['version'])
print('SHA-256:', IB['protocol']['sha256'])
print('Модель:', IB['provenance']['model_id'], '| политика:', IB['provenance']['policy_version'])
print('Привязка отчёта:', IDENTITY['status'], IDENTITY['experiment_id'])

rows = []
for mode in ('natural', 'no_keywords'):
    d = IB['deployed'][mode]
    rows.append({'режим': mode, 'n': d['n'], 'top1': d['top1_correct'],
                 'top1_accuracy': d['top1_accuracy'], 'top3': d['top3_correct'],
                 'macro_f1': d['macro_f1'], 'errors': d['errors']})
display(pd.DataFrame(rows).round(3))

print('OOD: теневых авторешений', IB['ood']['shadow_false_auto_n'], 'из', IB['ood']['n'],
      f"({IB['ood']['shadow_false_auto_rate']:.1%})")
display(pd.DataFrame(IB['criteria']['checks'].items(), columns=['критерий', 'пройден']))
print('Все заранее зафиксированные критерии:', IB['criteria']['all_passed'])
PE = IB.get('policy_evaluation') or {}
if PE.get('natural'):
    display(pd.DataFrame(PE['natural']['curve']).round(3))
    print('Операционный режим (auto_enabled=False):', PE['operational_natural']['automatic_n'],
          'авторешений из', PE['operational_natural']['n'])


In [ ]:
# @title Ячейка 60: Сравнение признаков и 100 размеченных примеров
RC = json.loads((REPORTS_DIR / 'representation_comparison.json').read_text(encoding='utf-8'))

def few(name, mode, regime='balanced'):
    block = (RC.get('few_shot') or {}).get(regime, {}).get(name)
    if block is None:
        block = RC.get('few_shot_100', {}).get(name)
    return None if block is None else block['aggregate'][mode]['mean_top1']

rows = []
for name, v in RC['full_train'].items():
    rows.append({'признаки': name,
                 'полный train natural': v['natural']['top1_accuracy'],
                 'полный train no_keywords': v['no_keywords']['top1_accuracy'],
                 '100 balanced natural': few(name, 'natural', 'balanced'),
                 '100 balanced no_keywords': few(name, 'no_keywords', 'balanced'),
                 '100 proportional natural': few(name, 'natural', 'proportional'),
                 '100 proportional no_keywords': few(name, 'no_keywords', 'proportional')})
display(pd.DataFrame(rows).round(3))
print('LLM:', RC['llm_zero_few_shot']['status'], '—', RC['llm_zero_few_shot'].get('reason', ''))
print('CV protocol:', RC.get('cv_protocol'))


In [ ]:
# @title Ячейка 61: Полная матрица потерь и ожидаемые потери
from support_router.business import cost_matrix, cost_sensitive_action, expected_losses

CM = cost_matrix()
display(pd.DataFrame(CM['matrix'], index=CM['classes'], columns=CM['actions']).round(2))

# Пример аудита: CONTACT 0.86, PAYMENT 0.14. Ошибка PAYMENT стоит 20 проверок.
classes = CM['classes']
p = [0.0] * len(classes)
p[classes.index('CONTACT')] = 0.86
p[classes.index('PAYMENT')] = 0.14
costs = {'PAYMENT': 20, 'CONTACT': 5}
actions, losses = expected_losses([p], classes, costs=costs, review_cost=1,
                                  default_misroute_cost=5)
display(pd.DataFrame(losses, columns=actions).round(2))
chosen, _, _ = cost_sensitive_action([p], classes, costs=costs, review_cost=1,
                                     default_misroute_cost=5)
print('Выбранное действие:', chosen[0], '— проверка дешевле ошибочного пропуска PAYMENT')

meta = json.loads((MODELS_DIR / 'router_meta.json').read_text(encoding='utf-8'))
shadow = meta['test']['shadow_routing']
print('Текущий релиз:', meta['model_id'], '| политика:', meta['policy']['version'])
print('Test Bitext, теневой режим:', shadow['automatic_n'], 'сам,',
      shadow['automatic_queue_errors'], 'ошибок очереди,', shadow['manual_n'], 'к оператору')


In [ ]:
# @title Ячейка 62: Калибровка по категориям, редкие классы и reliability
CAL = IB['calibration']['calibration_split']
rows = []
for cat, r in CAL['per_category'].items():
    rows.append({'категория': cat, 'support': r['support'], 'статус': r['support_status'],
                 'Brier': r['brier'], 'log_loss': r['log_loss'], 'ECE': r['ece'],
                 'precision': r['precision'], 'recall': r['recall']})
display(pd.DataFrame(rows).round(4))
print('Недостаточно данных для устойчивых выводов:', ', '.join(CAL['insufficient_support']) or 'нет')
print('Порог support:', CAL['min_support'])
REL = IB['calibration']['overall_calibration_split'].get('reliability', [])
if REL:
    display(pd.DataFrame(REL).round(3))
print('Источники калибровки:', ', '.join((IB['calibration'].get('by_source') or {}).keys()))


In [ ]:
# @title Ячейка 63: Каскад, риск-правила, несколько просьб и feedback
CASC = IB['cascade']
rows = []
for mode in ('natural', 'no_keywords'):
    c = CASC[mode]
    rows.append({'набор': mode, 'подсказок': c['suggestions_n'],
                 'точность подсказок': c['suggestion_precision'],
                 'исправил': c['corrected'], 'испортил': c['regressed']})
display(pd.DataFrame(rows).round(3))
print('OOD: ложных подсказок', CASC['ood']['false_hints'], 'из', CASC['ood']['n'],
      f"({CASC['ood']['false_hint_rate']:.1%})")
DET = CASC.get('detailed') or {}
if DET.get('natural'):
    display(pd.DataFrame([
        {'набор': mode, 'primary_accuracy': DET[mode]['primary_accuracy'],
         'final_accuracy': DET[mode]['final_accuracy'],
         'fallback_attempt_rate': DET[mode]['fallback_attempt_rate'],
         'suggestion_precision': DET[mode]['suggestion_precision'],
         'latency': DET[mode]['latency']}
        for mode in ('natural', 'no_keywords')
    ]).round(3))

RISK = IB['risk']['any_risk_flag']
print('Риски, frozen: precision', round(RISK['precision'], 3), '| recall', round(RISK['recall'], 3),
      '| нижняя граница Wilson', round(RISK['recall_wilson_lower_95'], 3))
if IB['risk'].get('development'):
    DEV = IB['risk']['development']['any_risk_flag']
    print('Риски, development: precision', round(DEV['precision'], 3), '| recall', round(DEV['recall'], 3))
MULTI = IB['multi_request']
print('Несколько просьб: точных совпадений', MULTI['exact_matches'], 'из', MULTI['n'],
      '| precision', round(MULTI['intent_precision'], 3), '| recall', round(MULTI['intent_recall'], 3))
if MULTI.get('per_intent'):
    display(pd.DataFrame(MULTI['per_intent']).T.round(3))

FB = json.loads((REPORTS_DIR / 'feedback_simulation.json').read_text(encoding='utf-8'))
for mode in ('natural', 'no_keywords'):
    perfect = FB['runs'][mode]['operator_error_0.0']['metrics']['measured_accuracy']
    noisy = FB['runs'][mode]['operator_error_0.2']['metrics']['measured_accuracy']
    print(f'Feedback {mode}: идеальный оператор {perfect:.1%}, ошибка оператора 20% {noisy:.1%}')


In [ ]:
# @title Ячейка 64: Релизный манифест и проверка версий
from support_router.identity import validate_report_identity

MAN = json.loads((REPORTS_DIR / 'release_manifest.json').read_text(encoding='utf-8'))
M = json.loads((MODELS_DIR / 'router_meta.json').read_text(encoding='utf-8'))
print('experiment_id:', MAN['experiment_id'])
print('model_id:', MAN['identity']['model_id'])
print('model_name:', MAN['identity']['model_name'])
print('policy_version:', MAN['identity']['policy_version'])
print('code_version:', MAN['identity']['code_version'])
print('commits:', MAN.get('commits'))
print('git_commit:', MAN['identity']['git_commit'], '| dirty:', MAN['git_dirty'])
print('raw_data_sha256:', MAN['identity']['raw_data_sha256'])
print('benchmark_protocol_sha256:', MAN['identity']['benchmark_protocol_sha256'])
print('Привязка манифеста:', validate_report_identity(M, MAN)['status'])
display(pd.DataFrame(MAN['evaluation_sets'].items(), columns=['набор', 'sha256']))
print('Отчёты:', ', '.join(MAN['reports']))


In [ ]:
# @title Ячейка 65: Выводы этапа 13 (динамически)
from IPython.display import Markdown

nat, nk = IB['deployed']['natural'], IB['deployed']['no_keywords']
casc = IB['cascade']
risk = IB['risk']['any_risk_flag']
checks = IB['criteria']['checks']
display(Markdown(f"""
### Выводы этапа 13

- `natural` top-1 {nat['top1_accuracy']:.1%}, `no_keywords` top-1 {nk['top1_accuracy']:.1%}.
- OOD: {IB['ood']['shadow_false_auto_n']} ложных теневых авторешений из {IB['ood']['n']} ({IB['ood']['shadow_false_auto_rate']:.1%}).
- Каскад исправил {casc['natural']['corrected']} и испортил {casc['natural']['regressed']} на `natural`, {casc['no_keywords']['corrected']} и {casc['no_keywords']['regressed']} на `no_keywords`.
- Риск-правила: recall {risk['recall']:.1%} на замороженном наборе; это не защитный контур.
- Критерии: {sum(bool(v) for v in checks.values())} из {len(checks)} пройдено.
- Автоотправка остаётся выключенной. Следующий шаг — новый независимый набор или реальные обращения с операторской разметкой.
"""))


# Этап 12. Документация и итоги

Описание проекта лежит в репозитории: `README.md` (запуск, результаты), `docs/architecture.md` (части системы и их связи), `docs/api.md` (HTTP API), `docs/runbook.md` (метрики, оповещения и что делать при сбоях), а также отчёт и презентация в `docs/`. После этапа 13 ячейка 55 собирает отчёт из текущего релиза и отдельно помечает исторический прогон 3.1.2, ячейка 56 упаковывает проект и проверяет архив распаковкой в пустую папку, ячейка 57 сверяет цифры отчёта и презентации с прогоном, ячейка 58 подводит итог проекта. Полная карта изменений: `docs/audit_change_map.md`.


In [ ]:
# @title Ячейка 55: Отчёт о результатах из единственного источника
# @markdown `reports/current_release.json` и `reports/results.md` собираются из JSON-отчётов и паспорта модели.
from IPython.display import Markdown
from support_router.reporting import write_current_release, write_results_markdown

CUR = write_current_release(PROJECT_DIR)
MD = write_results_markdown(PROJECT_DIR)
print('current_release:', CUR)
print('results.md:', MD, '| строк:', len(MD.read_text(encoding='utf-8').splitlines()))
display(Markdown(MD.read_text(encoding='utf-8')))


In [ ]:
# @title Ячейка 56: Архив проекта и проверка в чистой папке
# @markdown Собираем в один zip всё, что нужно для запуска, и проверяем его как посторонний человек:
# @markdown распаковываем в пустую папку, запускаем самопроверку приложения и все автотесты.
import zipfile, tempfile
ARCHIVE = PROJECT_DIR/'support-ticket-router.zip'
SKIP_PARTS = {'__pycache__', '.pytest_cache', '.ruff_cache', '.git', '.ipynb_checkpoints'}
ARCHIVE_DIRS = ['src', 'scripts', 'tests', 'docs', 'monitoring', 'notebooks', 'data/eval', '.github']
ARCHIVE_FILES = ['README.md', 'pyproject.toml', 'requirements.txt', 'requirements-api.txt', 'Dockerfile',
                 '.dockerignore', 'docker-compose.yml', '.gitignore', 'app.py',
                 'models/router.joblib', 'models/router_meta.json', 'models/encoder/encoder_manifest.json',
                 'data/processed/split_manifest.json', '.gitattributes']
ARCHIVE_FILES += ['models/encoder/' + name for name in ('model.onnx', 'tokenizer.json', 'LICENSE', 'NOTICE.md')]
# Include only the repository dataset. Other uploaded CSVs stay outside the archive.
bundled_raw = raw_path.resolve() == (PROJECT_DIR / DEFAULT_RAW).resolve()
if bundled_raw:
    ARCHIVE_FILES += [DEFAULT_RAW, 'data/raw/LICENSE.txt', 'data/raw/README.txt']
for folder in ARCHIVE_DIRS:
    ARCHIVE_FILES += sorted(str(p.relative_to(PROJECT_DIR)) for p in (PROJECT_DIR/folder).rglob('*')
                            if p.is_file() and not SKIP_PARTS & set(p.parts))
ARCHIVE_FILES += sorted(str(p.relative_to(PROJECT_DIR)) for p in REPORTS_DIR.glob('*')
                        if p.is_file() and p.suffix in {'.json', '.csv', '.md'} and p.stat().st_size < 3_000_000)
ARCHIVE_FILES += sorted(str(p.relative_to(PROJECT_DIR)) for p in (REPORTS_DIR/'figures').glob('*.png'))
ARCHIVE_FILES = list(dict.fromkeys(f for f in ARCHIVE_FILES if (PROJECT_DIR/f).exists()))


def build_archive():
    with zipfile.ZipFile(ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as zf:
        for rel in ARCHIVE_FILES:
            zf.write(PROJECT_DIR/rel, rel)
    return ARCHIVE.stat().st_size


build_archive()
clean_dir = Path(tempfile.mkdtemp(prefix='router_clean_'))
with zipfile.ZipFile(ARCHIVE) as zf:
    zf.extractall(clean_dir)
clean_env = dict(os.environ, PYTHONPATH=os.pathsep.join([str(clean_dir/'src'), str(clean_dir)]),
                 PYTEST_DISABLE_PLUGIN_AUTOLOAD='1', SUPPORT_SKIP_SEMANTIC_TESTS='1', GRADIO_ANALYTICS_ENABLED='False',
                 NO_PROXY='127.0.0.1,localhost', no_proxy='127.0.0.1,localhost')
clean_env.pop('SUPPORT_ROUTER_HOME', None)
clean_env.pop('SUPPORT_ROUTER_MODELS_DIR', None)
if bundled_raw:
    assert sc.sha256_file(clean_dir / DEFAULT_RAW) == split_manifest['raw_sha256']
    for notice in ('LICENSE.txt', 'README.txt'):
        assert (clean_dir / 'data/raw' / notice).is_file()
clean_check = subprocess.run([sys.executable, 'app.py', '--check'], cwd=clean_dir, env=clean_env,
                             capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=600)
assert clean_check.returncode == 0 and 'CHECK OK' in clean_check.stdout, clean_check.stderr[-3000:]
clean_tests = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--color=no', '--disable-warnings',
                              '-p', 'no:cacheprovider'], cwd=clean_dir, env=clean_env,
                             capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=1800)
assert clean_tests.returncode == 0, clean_tests.stdout[-3000:] + clean_tests.stderr[-3000:]
clean_summary = clean_tests.stdout.strip().splitlines()[-1]
print('Чистая папка:', clean_dir)
print('Самопроверка приложения: CHECK OK')
print('Автотесты из архива:', clean_summary, '(семантические пропущены, чтобы не скачивать энкодер заново)')

saved_results = json.loads((REPORTS_DIR/'results.json').read_text(encoding='utf-8'))
saved_results['stage12'] = {'archive': ARCHIVE.name, 'files': ARCHIVE_FILES,
                            'clean_folder_app_check': True, 'clean_folder_tests': clean_summary,
                            'results_report': 'reports/results.md'}
sc.write_json(REPORTS_DIR/'results.json', saved_results)
size = build_archive()  # пересобираем, чтобы в архиве был results.json с разделом stage12
print(f'Архив {ARCHIVE.name}: {len(ARCHIVE_FILES)} файлов, {size/1e6:.1f} МБ')
display(pd.Series([f for f in ARCHIVE_FILES if '/' not in f or f.split('/')[0] in {'src', 'monitoring', 'docs', 'models'}],
                  name='главные файлы архива'))
if 'google.colab' in sys.modules and os.environ.get('SUPPORT_NONINTERACTIVE') != '1':
    from google.colab import files
    files.download(str(ARCHIVE))


In [ ]:
# @title Ячейка 57: Сравнение исторических слайдов версии 3.1.1 с текущим прогоном
# @markdown Цифры на слайдах взяты из прогона, по которому собрана презентация. Здесь они сравниваются
# @markdown с тем, что получилось сейчас. Расхождение не ошибка ноутбука, а сигнал поправить слайд.
R = json.loads((REPORTS_DIR/'results.json').read_text(encoding='utf-8'))
S11 = R['stage11']
H, O, T11 = S11['hard_test'], S11['ood_new'], S11['tests']
TS = R['test_shadow_routing']
loio = {x['model']: x['all25_mean_recall'] for x in R['loio_summary']}
vs_base = next(x for x in R['loio_paired_intent_bootstrap'] if x['candidate'].startswith('lr_word'))
per_file = {row['файл']: row['всего'] for row in T11['per_file']}
n = lambda x: f'{x:,}'.replace(',', ' ')
p1 = lambda x: f'{x:.1%}'
SLIDES = [
    ('Данные', 'обращений в датасете', '21 534', n(R['data']['cleaning']['raw_rows'])),
    ('Данные', 'строк после очистки', '19 700', n(R['data']['cleaning']['clean_rows'])),
    ('Данные', 'удалено дублей', '1 834', n(R['data']['cleaning']['deduplicated_rows'])),
    ('Данные', 'групп похожих фраз', '3 646', n(R['data']['grouping']['n_groups'])),
    ('Данные', 'разбиение', '11 970 / 2 240 / 2 368 / 3 122',
     ' / '.join(n(R['data']['splits'][k]['rows']) for k in ['train', 'calibration', 'val', 'test'])),
    ('Модель', 'LOIO выбранной', '81.8%', p1(loio[R['model_name']])),
    ('Модель', 'LOIO базовой', '68.6%', p1(next(v for k, v in loio.items() if k.startswith('lr_word')))),
    ('Модель', 'интентов, где выбранная лучше базовой', '22', str(vs_base['candidate_losses'])),
    ('Три проверки качества', 'test macro-F1', '0.996', f"{R['test']['macro_f1']:.3f}"),
    ('Три проверки качества', 'ошибок на test', '10 из 3 122', f"{R['test']['errors']} из {n(R['test']['n'])}"),
    ('Три проверки качества', 'новые фразы: тема верна', '115 из 220 (52.3%)',
     f"{H['all']['model_correct']} из {H['all']['n']} ({p1(H['all']['model_accuracy'])})"),
    ('Три проверки качества', 'обычные слова: модель / базовая', '89.1% / 81.8%',
     f"{p1(H['natural']['model_accuracy'])} / {p1(H['natural']['baseline_accuracy'])}"),
    ('Три проверки качества', 'без подсказок: модель / базовая', '15.5% / 15.5%',
     f"{p1(H['no_keywords']['model_accuracy'])} / {p1(H['no_keywords']['baseline_accuracy'])}"),
    ('Порог и автоотправка', 'уверенность / сходство', '0.85 / 0.45',
     f"{R['policy']['confidence_threshold']} / {R['policy']['similarity_threshold']}"),
    ('Порог и автоотправка', 'test: верно / оператору / ошибка', '3 062 / 54 / 6',
     f"{n(TS['automatic_n'] - TS['automatic_queue_errors'])} / {TS['manual_n']} / {TS['automatic_queue_errors']}"),
    ('Порог и автоотправка', 'новые фразы: верно / оператору / ошибка', '65 / 154 / 1',
     f"{H['all']['shadow_auto_right_queue']} / {H['all']['shadow_manual']} / {H['all']['shadow_wrong_queue']}"),
    ('Порог и автоотправка', 'не по адресу: оператору / ошибка', '95 / 5',
     f"{O['n'] - O['shadow_false_auto']} / {O['shadow_false_auto']}"),
    ('Тестирование и найденная проблема', 'автотестов, известных пропусков', '222, 4',
     f"{T11['tests']}, {T11['xfailed_known_gaps']}"),
    ('Тестирование и найденная проблема', 'тестов HTTP API', '48', str(per_file['test_api.py'])),
    ('Тестирование и найденная проблема', 'тестов метрик и алертов', '6', str(per_file['test_monitoring.py'])),
    ('Тестирование и найденная проблема', 'упавших тестов', '0', str(T11['failures'] + T11['errors'])),
    ('Тестирование и найденная проблема', 'рабочий режим отправил сам', '0 из 320',
     f"{H['all']['deployed_auto'] + O['deployed_auto']} из {H['all']['n'] + O['n']}"),
    ('Тестирование и найденная проблема', 'нестандартных формулировок пройдено', '28 из 30',
     f"{S11['nonstandard_manual']['passed']} из {S11['nonstandard_manual']['n']}"),
    ('Тестирование и найденная проблема', 'фраз проекта / изменённых флагов', '21 689 / 0',
     f"{n(S11['fix']['project_texts_checked'])} / {S11['fix']['project_texts_changed']}"),
    ('Выводы и ограничения', 'без слов-подсказок тема верна', '15%', f"{H['no_keywords']['model_accuracy']:.0%}"),
]
api_example = sc.route_ticket('Someone hacked my account', model)
SLIDES += [
    ('Рефакторинг', 'файлов пакета', '19', str(len(list((PROJECT_DIR/'src'/'support_router').glob('*.py'))))),
    ('Рефакторинг', 'тестов команды', '88', str(per_file['test_support_core.py'])),
    ('Архитектура', 'правил алертов', '11',
     str(len(re.findall(r'^\s*- alert:', (PROJECT_DIR/'monitoring'/'alert_rules.yml').read_text(encoding='utf-8'), flags=re.M)))),
    ('HTTP API и Docker', 'Someone hacked my account', 'MANUAL_REVIEW, P1, POSSIBLE_FRAUD, ACCOUNT, Queue_Manual_Review',
     ', '.join([api_example['status'], api_example['priority'], *api_example['risk_flags'],
                str(api_example['category']), str(api_example['queue'])])),
]
check = pd.DataFrame(SLIDES, columns=['слайд', 'что', 'на слайде', 'в этом прогоне'])
check['совпадает'] = np.where(check['на слайде'] == check['в этом прогоне'], '✓', 'поправить слайд')
display(check)
mismatch = check[check['совпадает'] != '✓']
if mismatch.empty:
    print(f"Все {len(check)} {plural(len(check), 'цифра', 'цифры', 'цифр')} презентации совпадают с этим прогоном.")
else:
    print(f'Не совпадают {len(mismatch)} из {len(check)}: поправьте эти места на слайдах.')


In [ ]:
# @title Ячейка 58: Итог проекта
# @markdown Текст собирается из текущего релиза и независимого отчёта, а исторический прогон 3.1.2 помечен отдельно.
from IPython.display import Markdown
M = json.loads((MODELS_DIR / 'router_meta.json').read_text(encoding='utf-8'))
IB = json.loads((REPORTS_DIR / 'independent_benchmark.json').read_text(encoding='utf-8'))
MAN = json.loads((REPORTS_DIR / 'release_manifest.json').read_text(encoding='utf-8'))
shadow = M['test']['shadow_routing']
nat, nk = IB['deployed']['natural'], IB['deployed']['no_keywords']
p = lambda x: f'{x:.1%}'

final_md = f"""
## Итог проекта

За двенадцать этапов собрана система, которая читает обращение клиента, предлагает одну из 11 тем и
очередь, ставит приоритет и отмечает риски. После аудита добавлен этап 13: независимый замороженный
набор, cost-sensitive правило, калибровка по категориям, отдельные метрики каскада и контур обратной
связи оператора. Автоотправка остаётся выключенной.

| Проверка | Результат |
|---|---|
| Текущая модель | `{M['model_id']}` / `{M['model_name']}` |
| Версия кода / политика | {MAN['identity']['code_version']} / `{MAN['identity']['policy_version']}` |
| Коммиты | {MAN.get('commits')} |
| experiment_id | `{MAN['experiment_id']}` |
| Test Bitext | macro-F1 {M['test']['classification']['macro_f1']:.3f}, ошибок {M['test']['classification']['errors']} |
| Теневой режим test | {shadow['automatic_n']} сам, ошибок очереди {shadow['automatic_queue_errors']}, к оператору {shadow['manual_n']} |
| Независимый `natural` | {p(nat['top1_accuracy'])} top-1, {p(nat['macro_f1'])} macro-F1 |
| Независимый `no_keywords` | {p(nk['top1_accuracy'])} top-1, {p(nk['macro_f1'])} macro-F1 |
| OOD | {p(IB['ood']['shadow_false_auto_rate'])} ложных авторешений |
| Критерии аудита | {sum(IB['criteria']['checks'].values())} из {len(IB['criteria']['checks'])} пройдено |
| Автотесты | см. CI и локальный pytest; число не вводится вручную |
| Автоотправка | выключена; только теневой режим |

Главный вывод: почти идеальный Bitext не переносится на новые формулировки. Система полезна как
подсказка оператору, но не готова к самостоятельной маршрутизации. Следующий шаг — новый замороженный
набор или реальные обращения с операторской разметкой.
"""
display(Markdown(final_md))
